# Imitation Learning — bám làn + đèn tín hiệu (bước 2/3)

Warm-start cho policy network của DRL (bước 3, xem `drl_training/`).

**Input**: mask segmentation 4 lớp (one-hot) + vector số `speed_mps`, `speed_limit_kmh`,
`traffic_light_state` (one-hot 3 nhãn: yellow/red/unknown). `yaw_rate_rps`,
`previous_steer`, `previous_longitudinal` đã bị **loại** ở v9 — xem §2.3.
**Output**: `[steer, longitudinal]` ∈ [-1, 1] (longitudinal âm = phanh, dương = ga).

Hai hợp đồng phải giữ đúng, cả hai đều là loại lỗi **im lặng** (không có exception nào):

1. **Không gian nhãn** phải khớp `train-seg.ipynb`, `data_collection/carla_collector/schema.py`
   (`RAW_TO_TRAIN_LANE`) và `drl_training/policy/backbone.py` (`NUM_CLASSES`). Nếu số kênh
   trùng nhau mà bảng nhãn khác nhau thì `load_state_dict` không báo gì — model chỉ đọc sai
   kênh và lái sai. §2b tự đối chiếu với file `.pth` của seg.
2. **Observation** không được chứa `lane_offset_m`, `heading_error_rad`, `is_junction`: đó là
   các đại lượng dùng làm *reward* ở bước DRL. Để lọt vào input thì (a) model học cách đọc
   thẳng sai số thay vì nhìn ảnh (leakage), (b) DRL không có các cột này nên checkpoint
   warm-start lệch shape. Dataset vẫn trả chúng ra riêng dưới tên `aux` để chẩn đoán ở §13.

**Vì sao dữ liệu thu ở 5 FPS.** `previous_steer` là đặc trưng nguy hiểm nhất trong behavior
cloning: ở 20 FPS vô-lăng gần như không kịp đổi trong 50 ms nên `previous_steer ≈ steer`, và
model đạt loss rất thấp bằng cách chép lại nó, bỏ qua hoàn toàn ảnh segmentation. Lúc chạy
thật `previous_steer` là hành động của chính nó ở bước trước → sai số tích luỹ, xe trôi khỏi
làn. Lỗi này **không** hiện ra trong val loss, nên §6 đo baseline "chép lại" và §13 bắt buộc
model phải thắng nó.

Chia train/val theo **town** (Town01–04 / Town05) giống hệt notebook segmentation: Town05
chưa từng xuất hiện lúc train nên MAE sẽ cao hơn cách chia theo session — đó là con số thật.

> **Đã đồng bộ với `train-segment-lane.ipynb`** (lần chạy đạt `lane_mIoU = 0.8917`,
> epoch 17, Unet/ResNet34 + SCSE bias-free, 4 lớp `Background/Road/RoadLine/Sidewalk`
> ở 384×480). Những chỗ đã sửa: `DATASET_ROOT` (tự dò 1–4 tầng mount, mặc định trỏ
> đúng đường dẫn seg đã dùng), số frame kỳ vọng theo TỪNG town (35 000 / 5 000 chứ
> không phải 40 000 / 10 000), cách dò `best_carla_lane_seg.pth`, §5b dựng lại model
> seg theo `decoder_attention_type` + `scse_bias_free` đọc từ chính checkpoint, và
> `NUM_WORKERS`/`persistent_workers` theo đúng bài học rò RAM của notebook seg.
>
> Quyết định duy nhất cần chốt trước khi chạy: `USE_PREDICTED_SEGMENTATION` ở §2.


> **Bản v9 — sửa lỗi "checkpoint không dùng được".** Ba thay đổi so với v7, tất cả đều bắt
> nguồn từ việc chạy checkpoint v8 trên CARLA thật (`drl_training/demo_il.py`), nơi xe đứng
> im tại vạch xuất phát với tỉ lệ phanh 100%:
>
> 1. **Bỏ rò rỉ quan sát** (§2.3): `yaw_rate_rps` + `previous_steer` +
>    `previous_longitudinal` ra khỏi observation. Chúng được đo cùng bước thời gian với
>    hành động nên model đọc thẳng đáp án; trong vòng kín chúng thành vòng lặp tự duy trì.
>    `SCALAR_FEATURE_DIM` 8 → **5**.
> 2. **Thêm `aux_head`** (§10): buộc nhánh CNN dự đoán `lane_offset_m` +
>    `heading_error_rad` từ ảnh. Bỏ rò rỉ mới chỉ *cấm* đường tắt; đây là phần *tạo ra*
>    tín hiệu học từ ảnh.
> 3. **`POOL_GRID = (4, 6)`** thay cho `(1, 1)` (§2.4/§10): giữ bố cục trái-phải của làn.
>    Phải khớp `drl_training/policy/backbone.py` — bên đó đã đặt (4, 6) sẵn, và chính chỗ
>    lệch này khiến checkpoint v8 báo lỗi shape ở `cnn_fc.0.weight` lúc warm-start.
>
> Kèm theo: `NORMALIZE_ACTION_LOSS` (cân bằng gradient steer/longitudinal),
> `DROP_STATIONARY_RUNS = True` (chống điểm hút "speed≈0 → phanh" qua `speed_mps`),
> `SAMPLER_MODE = "none"` cho lần chạy đầu.
>
> Giữ nguyên toàn bộ cấu trúc v7: **trực quan hoá dữ liệu THÔ đặt trước bước lọc**
> (§6) để thấy bộ dữ liệu gốc, và mọi cell dài được chia nhỏ theo từng việc.
>
> Hai lỗi dữ liệu tìm ra ở lần chạy v6, đã sửa mặc định:
> 1. Luật *"frame trùng khít frame trước"* bỏ **7 152 frame (21.9%)** khỏi train — hầu hết là
>    frame dừng đèn đỏ. Tỉ lệ phanh train tụt còn **2.8%** trong khi val có **48.3%**. Đây
>    đúng là cơ chế đã làm hỏng longitudinal ở v4. Nay mặc định **tắt**.
> 2. `MAX_HEADING_ERROR = 1.0 rad` cắt mất frame **vào cua ở ngã tư**: trung bình `|steer|`
>    giảm **44%** sau lọc. Nay luật này **bỏ qua frame `is_junction`**.


## 1. Import

In [ ]:
!pip install segmentation_models_pytorch
import os, glob, random, time
from concurrent.futures import ThreadPoolExecutor

import cv2
# Kaggle GPU session chỉ có 4 vCPU. Để OpenCV tự spawn thread thì nó tranh CPU với
# DataLoader worker và với ThreadPoolExecutor dựng cache mask ở §8.
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Cấu hình

### 2.1 Đường dẫn dữ liệu & danh sách town

In [ ]:
def find_town_dirs(root, town):
    hits = sorted(h for h in glob.glob(os.path.join(root, f"{town}*")) if os.path.isdir(h))
    if not hits:
        raise FileNotFoundError(f"Không thấy folder '{town}*' trong {root}")
    return hits


def _first_dir_with(patterns, probe):
    for pat in patterns:
        for hit in sorted(glob.glob(pat)):
            if os.path.isdir(hit) and glob.glob(os.path.join(hit, probe)):
                return hit
    return None


# --- Dữ liệu -----------------------------------------------------------------------------
# Dò 1-4 tầng mount thay vì hardcode: hai notebook đọc hai bản dữ liệu khác nhau là lỗi im
# lặng, không có exception nào.
SEG_RUN_DATASET_ROOT = "/kaggle/input/datasets/dloc19/data-seg-il/CARLA_DATA"
_ROOT_PATTERNS = [SEG_RUN_DATASET_ROOT,
                  "/kaggle/input/data-seg-il/CARLA_DATA",
                  *[f"/kaggle/input/{'*/' * k}CARLA_DATA" for k in range(1, 5)]]
DATASET_ROOT = _first_dir_with(_ROOT_PATTERNS, "Town01*")
if DATASET_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy folder 'Town01*'. Mount hiện có: {sorted(glob.glob('/kaggle/input/*'))}")
if os.path.realpath(DATASET_ROOT) != os.path.realpath(SEG_RUN_DATASET_ROOT):
    print(f"[!] DATASET_ROOT = {DATASET_ROOT}\n"
          f"    seg đã chạy trên {SEG_RUN_DATASET_ROOT}. Nếu là hai bản dữ liệu khác nhau "
          f"thì dừng lại.")
else:
    print(f"DATASET_ROOT = {DATASET_ROOT}  (đúng bản train-seg đã dùng)")

TRAIN_TOWNS = ["Town01", "Town02", "Town03", "Town04"]
VAL_TOWNS   = ["Town05"]
TOWN_DIRS   = [(t, d) for t in TRAIN_TOWNS + VAL_TOWNS
               for d in find_town_dirs(DATASET_ROOT, t)]
print("Town dirs:")
for _t, _d in TOWN_DIRS:
    print("  -", _d)

TOWN_IMBALANCE_TOL = 0.25       # §6 so số frame từng town với trung vị

### 2.2 Không gian nhãn & độ phân giải

In [ ]:
# --- Không gian nhãn: khớp tuyệt đối train-seg.ipynb (§2b tự kiểm với .pth) --------------
CLASS_NAMES = ["Background", "Road", "RoadLine", "Sidewalk"]
NUM_CLASSES = len(CLASS_NAMES)
ROAD_ID     = CLASS_NAMES.index("Road")
ROADLINE_ID = CLASS_NAMES.index("RoadLine")
SKY_ID      = CLASS_NAMES.index("Sky") if "Sky" in CLASS_NAMES else None

# Mọi raw id không nhắc tên (trời 0/13, Vehicles 10, Pedestrian 4, Ground 14, Terrain 22...)
# rơi về Background. Ground -> Background chứ không phải Road: CARLA định nghĩa Ground là
# bục/vòng xuyến/sân, gán nó là Road tức dạy model rằng lề bê tông đi được.
RAW_TO_TRAIN = {6: 2, 7: 1, 8: 3, 16: 1}     # RoadLine, Road, Sidewalk, RailTrack->Road
SEG_LABEL_LUT = np.zeros(256, dtype=np.uint8)
for _raw, _train in RAW_TO_TRAIN.items():
    SEG_LABEL_LUT[_raw] = _train
del _raw, _train

PALETTE = np.array([[60, 60, 60], [128, 64, 128], [157, 234, 50], [244, 35, 232]], np.uint8)

# --- Độ phân giải ------------------------------------------------------------------------
SEG_NATIVE_HEIGHT, SEG_NATIVE_WIDTH = 384, 480
IMAGE_HEIGHT, IMAGE_WIDTH = 192, 240         # AdaptiveAvgPool -> không đổi shape checkpoint
THIN_COVER_THRESH = 0.25


def downscale_labels(lab, out_w=IMAGE_WIDTH, out_h=IMAGE_HEIGHT,
                     thin_ids=(ROADLINE_ID,), thr=THIN_COVER_THRESH):
    """NEAREST toàn cục, rồi khôi phục class mảnh bằng độ phủ diện tích.

    NEAREST thuần khi 384->192 xoá phần lớn vạch kẻ (rộng 2-3 px) — đúng tín hiệu quan
    trọng nhất cho bám làn. Chạy trên TRAIN ID nên dùng chung được cho cả ground-truth lẫn
    mask dự đoán; phải là cùng một hàm, nếu không phân phối train/inference lệch nhau.
    """
    if lab.shape[0] == out_h and lab.shape[1] == out_w:
        return lab
    out = cv2.resize(lab, (out_w, out_h), interpolation=cv2.INTER_NEAREST)
    for t in thin_ids:
        m = (lab == t)
        if not m.any():
            continue
        cov = cv2.resize(m.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
        out[cov > thr] = t
    return out

### 2.3 Hợp đồng observation với DRL

In [ ]:
# --- Hợp đồng observation với DRL --------------------------------------------------------
# =========================================================================================
# RÒ RỈ QUAN SÁT — thay đổi quan trọng nhất của v9. Đọc trước khi động vào danh sách này.
# =========================================================================================
# `yaw_rate_rps` ĐÃ BỊ LOẠI. Nó không phải một đặc trưng, nó là ĐÁP ÁN: xe đang quay CHÍNH
# VÌ vô-lăng đang quay, và hai đại lượng được đo ở CÙNG MỘT bước thời gian. Đo trên CARLA
# thật với chính checkpoint v4/v8 (giữ nguyên một khung hình, quét từng đầu vào một):
#
#     nguồn biến thiên       biên độ steer gây ra      so với ảnh
#     yaw_rate_rps           0.368 (v8) / 0.520 (v4)      114x / 290x
#     previous_steer         0.175                         54x
#     ẢNH segmentation       0.0032 (v8) / 0.0018 (v4)      1x
#
# Nhánh CNN gần như không đóng góp gì cho việc đánh lái. Trong vòng kín hậu quả là tất định:
# yaw_rate khởi tạo bằng 0 -> steer ~ 0 -> xe đi thẳng -> yaw_rate vẫn 0. Một vòng lặp tự
# duy trì khiến xe lao thẳng cho tới khi ra khỏi đường — đúng những gì đo được (v4 đi thẳng
# 47m rồi va chạm; v8 còn không rời được vạch xuất phát).
#
# `speed_mps` và `speed_limit_kmh` KHÔNG bị loại: tốc độ là hệ quả của ga TRONG QUÁ KHỨ, và
# là thông tin bắt buộc để quyết định ga hiện tại. Ranh giới là "cùng bước thời gian".
CONTINUOUS_COLS = ["speed_mps", "speed_limit_kmh"]

# Danh sách chốt chặn — §14 kiểm lại và ghi vào checkpoint để DRL đối chiếu được.
LEAKY_COLS = ["yaw_rate_rps", "previous_steer", "previous_longitudinal"]

# aux có HAI vai trò khác nhau, đừng lẫn:
#   AUX_COLS        : cột đọc từ CSV, dùng cho bảng chẩn đoán ở §13 (giữ nguyên như v7).
#   AUX_TARGET_COLS : tập con thực sự trở thành MỤC TIÊU HỌC PHỤ ở §8/§10/§11.
# Vẫn TUYỆT ĐỐI không đưa chúng vào observation — chúng là input của REWARD bên DRL. Làm
# MỤC TIÊU học phụ thì khác hẳn làm ĐẦU VÀO: model phải SUY RA chúng từ ảnh.
AUX_COLS        = ["lane_offset_m", "heading_error_rad", "is_junction"]   # chỉ để chẩn đoán
AUX_TARGET_COLS = ["lane_offset_m", "heading_error_rad"]                  # mục tiêu học phụ

# Collector chỉ ghi "green" ở 0.28% mẫu (tỉ số đỏ:xanh 79:1, hợp lý ~10:1) và val có 0 mẫu.
# Một ô one-hot gần như không được huấn luyện là bẫy im lặng: DRL sẽ bật nó lên lúc chạy
# thật và scalar_mlp trả ra giá trị không có cơ sở. Gộp vào "unknown" — cả hai cùng nghĩa
# "được phép đi". Đặt False sau khi collector được sửa.
MERGE_GREEN_INTO_UNKNOWN = True
TRAFFIC_LIGHT_RAW_VOCAB = ["green", "yellow", "red", "unknown"]     # để vẽ + thống kê
TRAFFIC_LIGHT_VOCAB = (["yellow", "red", "unknown"] if MERGE_GREEN_INTO_UNKNOWN
                       else list(TRAFFIC_LIGHT_RAW_VOCAB))          # để dựng one-hot
TRAFFIC_LIGHT_LABEL = {"green": "đèn xanh", "yellow": "đèn vàng",
                       "red": "đèn đỏ", "unknown": "không có đèn"}

# previous_* là con dao hai lưỡi: bật thì lệnh mượt nhưng model có thể phát lại chính đầu
# vào (§13 đo bằng baseline "chép"); tắt thì buộc đọc ảnh nhưng longitudinal mất tín hiệu
# chính vì mask chỉ có 4 lớp đường/vạch/vỉa hè, không chứa đèn hay xe khác.
# Tắt làm SCALAR_FEATURE_DIM đổi -> DRL PHẢI đọc `scalar_feature_order` từ checkpoint.
#
# v9: TẮT. Ngoài chuyện là đường tắt cho steer (biên độ 0.175 so với 0.003 của ảnh),
# `previous_longitudinal` còn tạo một điểm hút chết người ở chiều dọc. Đo trên CARLA thật
# với checkpoint v8_pre_on (`drl_training/runs/il_demo/`): model xuất phanh ngay bước đầu,
# giá trị đó quay lại làm ĐẦU VÀO của bước sau, và sau ~5 bước nó khoá cứng ở -0.99. Xe
# không bao giờ rời vạch xuất phát — 0.02 m trong 2 episode x 150 bước, tỉ lệ phanh 100%.
# Đây chính là lý do checkpoint v8 "không dùng được" dù val MAE rất đẹp: MAE đo trên quỹ
# đạo CỦA AUTOPILOT (open-loop), còn vòng kín thì đầu vào do chính model sinh ra.
USE_PREV_ACTIONS = False
PREV_ACTION_COLS = ["previous_steer", "previous_longitudinal"]
RAW_ACTION_COLS  = PREV_ACTION_COLS if USE_PREV_ACTIONS else []


def normalize_traffic_light(value):
    v = str(value).strip().lower()
    return v if v in TRAFFIC_LIGHT_VOCAB else "unknown"


SCALAR_FEATURE_DIM = len(CONTINUOUS_COLS) + len(RAW_ACTION_COLS) + len(TRAFFIC_LIGHT_VOCAB)
SCALAR_FEATURE_ORDER = CONTINUOUS_COLS + RAW_ACTION_COLS + \
    [f"traffic_light_{v}" for v in TRAFFIC_LIGHT_VOCAB]

# --- Bộ dữ liệu 5 FPS --------------------------------------------------------------------
# CONTROL_DT là hợp đồng với DRL: `previous_*` nghĩa là "lệnh của 1 bước trước". Đặt
# fixed_delta_seconds của CARLA đúng bằng giá trị này.
COLLECT_FPS = 5.0
CONTROL_DT  = 1.0 / COLLECT_FPS
EPISODE_COL = "session_id"

# --- Recovery ----------------------------------------------------------------------------
# 13.8% mẫu val lệch >0.15m nhưng gây 64% tổng sai số (MAE gấp ~8-11 lần vùng giữa làn).
# Autopilot luôn chạy giữa làn nên model chưa từng học cách quay về, và trong vòng kín hễ
# bắt đầu trôi là rơi vào đúng vùng nó yếu nhất -> sai số tự khuếch đại.
RECOVERY_OFFSET_THRESH = 0.15

### 2.4 Siêu tham số huấn luyện & lấy mẫu

In [ ]:
# --- Nhật ký thí nghiệm ------------------------------------------------------------------
RUN_NAME    = "v9_no_leak_aux"
RESULTS_LOG = "/kaggle/working/il_results.csv"

# --- Optimization ------------------------------------------------------------------------
# batch 128: model 0.146M tham số + mask trong RAM nên bs=32 để GPU rảnh, và val MAE dao
# động mạnh hơn cả khoảng cách tới baseline. LR scale theo căn bậc hai tỉ lệ batch.
BASE_LR, BASE_BATCH = 1e-3, 32
BATCH_SIZE     = 128
LEARNING_RATE  = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
EPOCHS         = 40
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 1.0
WARMUP_STEPS   = 200
MIN_LR_RATIO   = 0.02
EARLY_STOP_PATIENCE = 12

STEER_LOSS_WEIGHT = 2.0
LONGITUDINAL_LOSS_WEIGHT = 1.0

# --- Loss phụ: bắt nhánh CNN phải mã hoá vị trí ngang ------------------------------------
# Bỏ rò rỉ (§2.3) mới chỉ CẤM model đi đường tắt; nó không tự tạo ra tín hiệu học từ ảnh.
# Đây là phần TẠO ra tín hiệu đó: buộc nhánh CNN — và CHỈ nhánh CNN, xem `aux_head` ở §10,
# nó cắm thẳng vào đặc trưng ảnh và không thấy scalar — dự đoán xe đang lệch bao nhiêu mét
# so với tâm làn và lệch hướng bao nhiêu radian. Hai đại lượng đó CHỈ đọc được từ ảnh, nên
# gradient của chúng không có đường tắt nào để đi.
# Nhãn đã sẵn có: collector vẫn luôn ghi `lane_offset_m` / `heading_error_rad`; v7 chỉ dùng
# chúng để in bảng chẩn đoán ở §13. Không phải thu thêm dữ liệu.
AUX_LOSS_WEIGHT   = 0.5      # 0.0 = tắt hẳn, quay về hành vi v7
AUX_TARGET_SCALE  = [1.0, 0.3]   # đưa mét và radian về cùng thang trước khi tính loss

# --- Cân bằng thang đo giữa hai chiều hành động ------------------------------------------
# |steer| điển hình 0.005-0.03 còn |longitudinal| chạy cả dải [-1, 1]. Với cùng một beta,
# gradient của chiều longitudinal áp đảo, và "xuất steer ~ 0" trở thành cực tiểu rẻ nhất —
# chính xác thứ v4 rơi vào. Chia sai số mỗi chiều cho độ lệch chuẩn của chính nó (tính trên
# TRAIN ở §10) để hai chiều đóng góp gradient tương đương.
# Không đổi không gian ĐẦU RA -> `drl_training` không phải sửa gì.
NORMALIZE_ACTION_LOSS = True
# §13 chấm bằng MAE, mà SmoothL1 là bậc hai khi |err| < beta. beta=0.15 đặt gần như toàn bộ
# dữ liệu (80% mẫu đi thẳng) vào vùng gradient tỉ lệ sai số -> sai số nhỏ không tạo áp lực
# học. beta nhỏ đưa chúng về vùng tuyến tính, khớp metric, vẫn giữ tính bền cho cua gấp.
HUBER_BETA = 0.02

USE_EMA        = True
EMA_DECAY      = 0.999
EMA_EVAL_START = 3

# --- Cân bằng lại tập TRAIN (val Town05 không bao giờ bị đụng vào) -----------------------
#   none          : phân phối train = val, cơ hội cao nhất vượt baseline ở §13
#   steer         : cân bằng theo |steer| — hầu như không đổi tỉ lệ mẫu lệch làn
#   offset        : cân bằng theo |lane_offset_m| — 13.8% -> ~41% mẫu lệch làn mỗi epoch
#   steer+offset  : kết hợp; thêm "+tl" nếu muốn cân bằng cả đèn
# v9 dùng "none" cho lần chạy đầu: sau khi bỏ rò rỉ, phân phối train = phân phối val cho
# con số sạch nhất để so với baseline ở §13. Đổi sang "offset" chỉ khi §13 cho thấy MAE
# vùng lệch làn vẫn gấp nhiều lần MAE vùng giữa làn.
SAMPLER_MODE       = "none"
SAMPLER_POWER      = 0.5     # w = (1/tần_suất)**power; 1.0 đẩy tỉ số lên 23x, quá mạnh
SAMPLER_WEIGHT_CAP = 4.0     # trần tính theo "số lần so với bình thường" sau chuẩn hoá
OFFSET_BIN_EDGES  = [-1e-9, 0.15, 0.4, 0.8, np.inf]
OFFSET_BIN_LABELS = ["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"]

# Cờ này từng kéo tỉ lệ đèn đỏ trong train từ 27.8% xuống 4.8% trong khi val có 41.1%, làm
# longitudinal MAE nhảy 0.0221 -> 0.1919 (v6_r1). Nó bỏ frame dư thừa nhưng bỏ luôn cả
# HÀNH VI phanh. §6.2 đo lệch phân phối train/val và sẽ cảnh báo nếu tái diễn.
#
# v9 BẬT LẠI cờ này, và lý do đã đổi hẳn so với v7. Sau khi bỏ `previous_longitudinal` khỏi
# observation (§2.3), tín hiệu chi phối chiều dọc chỉ còn `speed_mps`. Nếu tập train đầy
# frame dừng đèn đỏ thì model học đúng một luật: "speed ~ 0 -> phanh". Mà xe LUÔN spawn ở
# tốc độ 0. Kết quả là một điểm hút y hệt lỗi của v8 (xe không rời vạch xuất phát), chỉ
# khác đường dẫn: qua `speed_mps` thay vì qua `previous_longitudinal`.
#
# STATIONARY_KEEP = 5 frame (1 giây ở 5 FPS) mỗi lần dừng vẫn đủ dạy "dừng khi đèn đỏ",
# đồng thời GIỮ được các frame KHỞI HÀNH từ trạng thái đứng yên — thứ model bắt buộc phải
# học nếu muốn tự cất bánh. §6.2 đo trực tiếp xem tỉ lệ phanh train/val có còn lệch không.
DROP_STATIONARY_RUNS = True
STATIONARY_SPEED     = 0.1
STATIONARY_KEEP      = 5

# --- Kiến trúc: lưới pooling --------------------------------------------------------------
# (1, 1) — bản v7 — là TRUNG BÌNH TOÀN ẢNH của từng kênh: nó bóp bản đồ đặc trưng 12x15
# thành một số mỗi kênh, nên phân bố TRÁI-PHẢI của làn đường (thứ duy nhất nói cho xe biết
# nó đang lệch về bên nào) phải đi vòng qua tương quan giữa các kênh thay vì được biểu diễn
# trực tiếp. (4, 6) giữ lại bố cục thô: 4 hàng (gần -> xa) x 6 cột (trái -> phải), tức
# `cnn_fc` nhận 64*24 = 1536 chiều thay vì 64.
#
# PHẢI KHỚP `POOL_GRID` trong drl_training/policy/backbone.py. Lệch một bên mà quên bên kia
# thì `il_compat.load_matching` báo lỗi shape ngay ở `cnn_fc.0.weight` — cố ý để như vậy,
# vì đây chính xác là loại lệch phải chết to chứ không được chạy im lặng. §10 có assert.
POOL_GRID = (4, 6)

### 2.5 Runtime & checkpoint segmentation

In [ ]:
# --- Runtime -----------------------------------------------------------------------------
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
N_GPU   = torch.cuda.device_count()

# NUM_WORKERS = 0 khi đã cache: __getitem__ chỉ index vào numpy trong RAM, rẻ hơn chi phí
# pickle qua IPC. Ngoài ra `_common` dùng chung cho hai loader + persistent_workers=True là
# đúng cái làm train-seg rò +2.9 GB/epoch.
CACHE_MASKS_IN_RAM = True
NUM_WORKERS = 0 if CACHE_MASKS_IN_RAM else min(os.cpu_count() or 2, 4)
SEG_INFER_WORKERS = min(os.cpu_count() or 2, 4)

STEER_CHECKPOINT_PATH = "/kaggle/working/best_il_model.pth"

# --- Checkpoint segmentation -------------------------------------------------------------
_SEG_PATTERNS = ["best_carla_lane_seg.pth",
                 "/kaggle/working/best_carla_lane_seg.pth",
                 *[f"/kaggle/input/{'*/' * k}best_carla_lane_seg.pth" for k in range(1, 5)]]
SEG_CHECKPOINT_PATH = next((h for pat in _SEG_PATTERNS for h in sorted(glob.glob(pat))
                            if os.path.isfile(h)), "best_carla_lane_seg.pth")
REQUIRE_SEG_CONTRACT = True     # thiếu .pth thì DỪNG thay vì bỏ qua kiểm tra nhãn

# True khớp đúng phân phối DRL gặp lúc chạy thật (Sidewalk IoU 0.81, RoadLine 0.89) nhưng
# tốn ~20-30 phút cho 40k ảnh x 2 forward và ~300 MB trong /kaggle/working.
USE_PREDICTED_SEGMENTATION = False
PREDICTED_MASK_DIR = "/kaggle/working/predicted_seg_masks"
SEG_PATHS_ARE_TRAIN_IDS = False


def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


DATALOADER_GENERATOR = torch.Generator()
DATALOADER_GENERATOR.manual_seed(SEED)

print(f"\n{NUM_CLASSES} class {CLASS_NAMES} | {IMAGE_HEIGHT}x{IMAGE_WIDTH} "
      f"(gốc {SEG_NATIVE_HEIGHT}x{SEG_NATIVE_WIDTH}) | scalar_dim={SCALAR_FEATURE_DIM}")
print(f"batch={BATCH_SIZE} | epochs={EPOCHS} | lr={LEARNING_RATE:.2e} | workers={NUM_WORKERS} "
      f"| cache_mask={CACHE_MASKS_IN_RAM} | amp={USE_AMP} | {N_GPU} GPU")
print(f"RUN_NAME={RUN_NAME} | huber_beta={HUBER_BETA} | ema={USE_EMA} "
      f"| sampler={SAMPLER_MODE}(p={SAMPLER_POWER}, cap {SAMPLER_WEIGHT_CAP}) "
      f"| prev_actions={USE_PREV_ACTIONS} | drop_stationary={DROP_STATIONARY_RUNS} "
      f"| pool={POOL_GRID} | aux_w={AUX_LOSS_WEIGHT}")
print(f"train towns={TRAIN_TOWNS} | val towns={VAL_TOWNS} "
      f"(§6 tự đếm và so với trung vị, dung sai {TOWN_IMBALANCE_TOL:.0%})")
print(f"seg ckpt = {SEG_CHECKPOINT_PATH}"
      + ("" if os.path.exists(SEG_CHECKPOINT_PATH) else "   [CHƯA THẤY FILE]"))
print(f"segmentation dùng để train = "
      f"{'DỰ ĐOÁN (§5b)' if USE_PREDICTED_SEGMENTATION else 'GROUND-TRUTH'}")

## 2b. Kiểm tra hợp đồng nhãn với model segmentation

Chốt chặn quan trọng nhất của notebook: bảng nhãn từng bị chép tay ở ba nơi (notebook seg,
notebook IL, `schema.py`) và đã lệch nhau trong thực tế. Cell này đối chiếu với chính file
`.pth` thay vì tin vào comment.

In [ ]:
SEG_META = {}
if os.path.exists(SEG_CHECKPOINT_PATH):
    _sc = torch.load(SEG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    _names = list(_sc["class_names"])
    _lut = np.array(_sc["label_lut"], dtype=np.int64)

    print(f"seg checkpoint : {SEG_CHECKPOINT_PATH}")
    print(f"  epoch        : {_sc.get('epoch', -1) + 1} "
          f"(trọng số '{_sc.get('weights_source', '?')}')")
    print(f"  lớp          : {_names}")
    print(f"  kiến trúc    : {_sc.get('model_arch')}/{_sc.get('encoder_name')} "
          f"| attention={_sc.get('decoder_attention_type')} "
          f"| scse_bias_free={_sc.get('scse_bias_free')}")
    print(f"  kích thước   : {_sc.get('image_height')}x{_sc.get('image_width')}")
    print(f"  lane_mIoU    : {_sc.get('best_lane_miou', float('nan')):.4f} "
          f"| drivable IoU {_sc.get('iou_drivable', float('nan')):.4f}")
    if _sc.get("per_class_iou"):
        print("  IoU/class    : " + ", ".join(
            f"{n}={v:.3f}" for n, v in zip(_names, _sc["per_class_iou"])))

    assert _names == CLASS_NAMES, (
        f"LỆCH TÊN/THỨ TỰ LỚP.\n  seg: {_names}\n  IL : {CLASS_NAMES}\n"
        "Sửa CLASS_NAMES ở §2 cho khớp seg, ĐỪNG sửa ngược lại.")

    _diff = [(r, int(_lut[r]), int(SEG_LABEL_LUT[r])) for r in range(23)
             if int(_lut[r]) != int(SEG_LABEL_LUT[r])]
    assert not _diff, ("LỆCH LUT tại (raw, seg, IL): " + str(_diff) +
                       "\nThường gặp: seg đang để IGNORE_UNLABELED=True nên raw 0 -> 255, "
                       "hoặc AUTO_PRUNE đã gộp một class về Background.")

    # Bước thời gian: seg không dùng tới nó, nhưng nếu hai notebook đọc hai bản dataset khác
    # nhau thì đây là chỗ lộ ra sớm nhất.
    _fps = _sc.get("collect_fps")
    if _fps is not None and abs(_fps - COLLECT_FPS) > 1e-6:
        raise RuntimeError(f"LỆCH FPS: seg={_fps}, IL={COLLECT_FPS}. "
                           "Hai notebook đang đọc hai bản dataset khác nhau.")
    if bool(_sc.get("sky_as_class")) != ("Sky" in CLASS_NAMES):
        raise RuntimeError(f"Lệch cách xử lý bầu trời: seg sky_as_class="
                           f"{_sc.get('sky_as_class')}, IL {'có' if SKY_ID is not None else 'không có'}"
                           " class 'Sky'. Cả hai phải cùng gộp trời vào Background.")

    # Độ phân giải GỐC phải khớp: seg đọc mask ở 384x480 rồi train ở đó, IL đọc CÙNG file
    # PNG đó rồi hạ xuống IMAGE_HEIGHT/WIDTH. Lệch nghĩa là hai notebook nhìn hai bản mask
    # khác nhau, và §5b sẽ sinh mask ở sai kích thước.
    _sh, _sw = _sc.get("image_height"), _sc.get("image_width")
    if (_sh, _sw) != (SEG_NATIVE_HEIGHT, SEG_NATIVE_WIDTH):
        raise RuntimeError(f"LỆCH ĐỘ PHÂN GIẢI GỐC: seg train ở {_sh}x{_sw}, IL khai báo "
                           f"SEG_NATIVE = {SEG_NATIVE_HEIGHT}x{SEG_NATIVE_WIDTH}. "
                           "Sửa SEG_NATIVE_* ở §2 cho khớp seg.")

    # Cùng ngưỡng bảo tồn class mảnh -> vạch kẻ sống sót qua resize theo cùng một luật.
    _thr = _sc.get("thin_cover_thresh")
    if _thr is not None and abs(_thr - THIN_COVER_THRESH) > 1e-9:
        raise RuntimeError(f"LỆCH thin_cover_thresh: seg={_thr}, IL={THIN_COVER_THRESH}.")

    # §5b cần đúng những key này để dựng lại model; thiếu -> dừng ngay thay vì để
    # load_state_dict báo missing/unexpected key giữa chừng.
    if USE_PREDICTED_SEGMENTATION:
        _need = ["model_state_dict", "model_arch", "encoder_name",
                 "decoder_attention_type", "num_classes"]
        _lack = [k for k in _need if k not in _sc]
        if _lack:
            raise KeyError(f"Checkpoint seg thiếu key {_lack} — không dựng lại được model "
                           "cho §5b. Train lại seg bằng bản notebook mới nhất.")

    SEG_META = {k: _sc[k] for k in (
        "num_classes", "class_names", "model_arch", "encoder_name",
        "decoder_attention_type", "scse_bias_free", "image_height", "image_width",
        "norm_mean", "norm_std") if k in _sc}

    print(f"\nHợp đồng nhãn khớp: LUT, tên lớp, thứ tự index, FPS, độ phân giải gốc và "
          f"ngưỡng class mảnh đều giống nhau.")
    del _sc, _names, _lut, _diff
else:
    _msg = ("Không thấy checkpoint segmentation. Đã dò:\n"
            + "\n".join("    " + p for p in _SEG_PATTERNS))
    if REQUIRE_SEG_CONTRACT or USE_PREDICTED_SEGMENTATION:
        raise FileNotFoundError(
            _msg + "\n  Add output của train-seg.ipynb (best_carla_lane_seg.pth) làm input "
                   "dataset của notebook này.\n  Hoặc đặt REQUIRE_SEG_CONTRACT = False ở §2 "
                   "để train bằng ground-truth mà BỎ QUA kiểm tra hợp đồng nhãn.")
    print("[!] " + _msg)
    print("    Train bằng ground-truth vẫn chạy được, nhưng không ai xác nhận rằng hai model")
    print(f"    dùng chung bảng nhãn {CLASS_NAMES} — đó là loại lỗi im lặng.")

## 3. Nạp dữ liệu

### 3.1 Đọc il_fields.csv

In [ ]:
REQUIRED_COLS = ["session_id", "seg_label_path", "speed_mps", "yaw_rate_rps",
                 "previous_steer", "previous_longitudinal", "lane_offset_m",
                 "heading_error_rad", "speed_limit_kmh", "traffic_light_state",
                 "is_junction", "steer", "longitudinal"]
USED_COLS = set(REQUIRED_COLS) | {"frame"}       # il_fields.csv có ~120 cột


def load_town_il_csv(town_dir, town):
    df = pd.read_csv(os.path.join(town_dir, "il_fields.csv"),
                     usecols=lambda c: c in USED_COLS)
    # Đường dẫn trong CSV là tương đối theo thư mục Town -> đổi sang tuyệt đối.
    df["seg_label_path"] = town_dir + os.sep + df["seg_label_path"]
    df["rgb_path"] = [os.path.join(town_dir, "rgb", f"{int(f):08d}.png") for f in df["frame"]]
    df["town"] = town
    return df


df = pd.concat([load_town_il_csv(d, t) for t, d in TOWN_DIRS], ignore_index=True)
df = df.sort_values([EPISODE_COL, "frame"], kind="mergesort").reset_index(drop=True)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"CSV thiếu cột: {missing}. Cột hiện có: {list(df.columns)}"

# Bước thời gian thật, đo từ dữ liệu chứ không tin COLLECT_FPS. Thu ở tần số dày hơn khai
# báo là thứ biến previous_steer thành đường tắt (xem §6).
_gap = df.groupby(EPISODE_COL)["frame"].diff().dropna()
print(f"Khoảng cách frame giữa hai mẫu: trung vị {_gap.median():.0f}, "
      f"p10={_gap.quantile(.1):.0f}, p90={_gap.quantile(.9):.0f}")

### 3.2 Trạng thái đèn tín hiệu — chẩn đoán collector

In [ ]:
# --- Đèn tín hiệu ------------------------------------------------------------------------
# Giữ bản gốc 4 trạng thái để vẽ/thống kê; cột dùng cho model thì gộp theo vocab.
df["traffic_light_raw"] = (df["traffic_light_state"].astype(str).str.strip().str.lower()
                           .where(lambda s: s.isin(TRAFFIC_LIGHT_RAW_VOCAB), "unknown"))
_raw_tl = df["traffic_light_raw"].value_counts()
print("\ntraffic_light_state thô:")
print(_raw_tl.to_string())

# Nếu "unknown" có tốc độ trung bình cao thì đèn xanh đang nằm lẫn trong đó, tức collector
# chỉ ghi trạng thái khi xe giảm tốc.
print("\nTrung bình theo trạng thái đèn thô (dùng để đoán lỗi collector):")
print(df.groupby("traffic_light_raw")[["speed_mps", "longitudinal", "is_junction"]]
        .mean().round(3).to_string())
_n_red, _n_grn = int((_raw_tl.get("red", 0))), int(_raw_tl.get("green", 0))
if _n_grn:
    print(f"tỉ số đỏ:xanh = {_n_red/_n_grn:.1f} : 1   (hợp lý ~10:1 ở 5 FPS)")

df["traffic_light_state"] = df["traffic_light_state"].map(normalize_traffic_light)
if MERGE_GREEN_INTO_UNKNOWN:
    assert "green" not in TRAFFIC_LIGHT_VOCAB
    print(f"\nĐã gộp {_n_grn} mẫu 'green' vào 'unknown' -> vocab {TRAFFIC_LIGHT_VOCAB}, "
          f"scalar_dim={SCALAR_FEATURE_DIM}")
    print("  DRL phải map green -> unknown theo traffic_light_vocab đọc từ checkpoint.")

print(f"\nTổng {len(df)} mẫu, {df[EPISODE_COL].nunique()} session, "
      f"{len(df)/COLLECT_FPS/60:.0f} phút lái ở {COLLECT_FPS:.0f} FPS")
print(df.groupby("town").size().to_string())

_green_share = _n_grn / max(len(df), 1)
if _green_share < 0.05:
    print(f"\n[!] Đèn xanh chỉ chiếm {100*_green_share:.2f}% số mẫu — không thể là phân bố "
          f"lái xe bình thường (pha xanh dài hơn pha vàng nhiều lần).")
    print("    Gần như chắc là lỗi collector khi đọc traffic_light_state. Hệ quả: ô one-hot")
    print("    'green' gần như không được huấn luyện, và val không đánh giá được phần đèn.")
df.head()

## 4. Trực quan hoá dữ liệu thô

Vẽ **trước** mọi bước lọc, trên toàn bộ `df`. Đây là bộ dữ liệu như nó được thu
thập — hình dùng cho báo cáo nên lấy ở đây, không lấy sau khi lọc.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["steer"], bins=60)
axes[0].set_yscale("log")
axes[0].set_title("Phân bố Steer (dữ liệu thô)")
axes[0].set_xlabel("steer"); axes[0].set_ylabel("số frame (log)")

axes[1].hist(df["longitudinal"], bins=60)
axes[1].set_yscale("log")
axes[1].set_title("Phân bố Longitudinal (dữ liệu thô)")
axes[1].set_xlabel("longitudinal")

# Vẽ đủ 4 trạng thái từ cột thô, kể cả green đã bị gộp khi dựng one-hot: biểu đồ mô tả DỮ
# LIỆU, không mô tả vocab của model. Thang log vì green ~1e2 còn unknown ~1e4.
tl_counts = (df["traffic_light_raw"].value_counts()
             .reindex(TRAFFIC_LIGHT_RAW_VOCAB).fillna(0).astype(int))
axes[2].bar([TRAFFIC_LIGHT_LABEL[k] for k in tl_counts.index], tl_counts.values,
            color=["#2ca02c", "#e8a33d", "#d62728", "#8c9196"])
axes[2].set_yscale("log")
axes[2].set_ylim(1, max(tl_counts.max() * 6, 10))
axes[2].set_ylabel("số frame (log)")
axes[2].set_title("Trạng thái đèn tín hiệu (dữ liệu thô)")
axes[2].tick_params(axis="x", rotation=20)
for i, v in enumerate(tl_counts.values):
    axes[2].text(i, max(v, 1), f"{v:,}\n{100*v/max(tl_counts.sum(), 1):.2f}%",
                 ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("bieu_do_data_tho.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["speed_mps"], bins=60)
axes[0].set_title("Tốc độ (m/s)")
axes[1].hist(df["lane_offset_m"], bins=80)
axes[1].set_yscale("log")
axes[1].axvline(-RECOVERY_OFFSET_THRESH, color="r", ls=":", lw=1)
axes[1].axvline(RECOVERY_OFFSET_THRESH, color="r", ls=":", lw=1,
                label=f"±{RECOVERY_OFFSET_THRESH}m")
axes[1].set_title("Lệch tâm làn (m)"); axes[1].legend(fontsize=8)
axes[2].hist(df["heading_error_rad"], bins=80)
axes[2].set_yscale("log")
axes[2].set_title("Sai số hướng (rad)")
plt.tight_layout()
plt.savefig("bieu_do_data_tho_aux.png", dpi=300, bbox_inches="tight")
plt.show()

_brake = float((df["longitudinal"] < -0.1).mean())
_recov = float((df["lane_offset_m"].abs() > RECOVERY_OFFSET_THRESH).mean())
print(f"Dữ liệu thô: {len(df):,} frame | phanh {100*_brake:.1f}% "
      f"| lệch >{RECOVERY_OFFSET_THRESH}m {100*_recov:.1f}% "
      f"| |steer| trung bình {df['steer'].abs().mean():.4f}")
print("Ghi lại ba con số này: §7 sẽ in lại sau khi lọc, và mọi thay đổi lớn ở chúng nghĩa là")
print("một luật lọc đang cắt mất HÀNH VI chứ không chỉ bản ghi hỏng.")

## 5. Tiền xử lý — loại bản ghi bất thường

### 5.1 Ngưỡng

In [ ]:
# Luật TOÀN VẸN (hỏng thật) áp cho cả train lẫn val — giữ bản ghi hỏng trong val là chấm
# điểm model bằng câu hỏi không có đáp án. Luật DƯ THỪA chỉ áp cho train.
# Lọc val LÀM ĐỔI baseline ở §8 — đừng so thẳng steer_mae giữa lần có lọc và không.
CLEAN_ENABLED = True
CLEAN_VAL_TOO = True

MAX_LANE_OFFSET   = 2.0    # m. Làn CARLA rộng ~3.5m -> lệch >2m gần như luôn do tra cứu
                           # waypoint sai ở ngã tư, không phải xe thật sự ở đó.
MAX_HEADING_ERROR = 2.0    # rad (~57°)
# Ở ngã tư, waypoint tham chiếu lệch hướng xe rất nhiều trong khi xe đang cua HOÀN TOÀN
# ĐÚNG. Lần chạy v6 áp luật này cho mọi frame và cắt mất phần lớn dữ liệu vào cua: trung
# bình |steer| giảm 44% sau lọc. Bỏ qua frame ngã tư.
HEADING_RULE_SKIP_JUNCTION = True

MAX_SPEED          = 50.0  # m/s
VALID_SPEED_LIMITS = None  # vd {30., 40., 60., 90.}; None = bỏ qua
MAX_FRAME_GAP_MULT = 3.0   # bội số trung vị khoảng cách frame
SETTLE_FRAMES      = 10    # frame đầu mỗi session (spawn, xe chưa ổn định)
CHECK_MASK_EXISTS  = True

# Frame đứng yên tại đèn đỏ có steer/longitudinal/speed/offset trùng khít frame trước, nên
# luật này ăn thẳng vào chúng. Ở v6 nó bỏ 7 152 frame (21.9%) CHỈ của train, kéo tỉ lệ phanh
# train xuống 2.8% trong khi val có 48.3% — chính là cơ chế làm hỏng longitudinal ở v4.
# Nó bỏ frame dư thừa nhưng bỏ luôn cả HÀNH VI phanh. Mặc định TẮT.
DROP_DUPLICATE_FRAMES = False

### 5.2 Hàm lọc

In [ ]:
def clean_dataframe(frame, integrity_only=False, tag=""):
    """Trả về (frame đã lọc, bảng báo cáo). Không sửa gì tại chỗ."""
    n0 = len(frame)
    frame = frame.sort_values([EPISODE_COL, "frame"], kind="mergesort").reset_index(drop=True)
    drop, report = pd.Series(False, index=frame.index), []

    def rule(name, mask, integrity=True):
        nonlocal drop
        if integrity_only and not integrity:
            return
        # NaN trong điều kiện so sánh -> coi là bất thường, không phải "bỏ qua".
        mask = pd.Series(np.asarray(mask), index=frame.index).fillna(True).astype(bool)
        new = mask & ~drop           # chỉ đếm phần MỚI bị loại
        report.append((name, int(new.sum())))
        drop = drop | new

    num_cols = CONTINUOUS_COLS + PREV_ACTION_COLS + AUX_COLS + ["steer", "longitudinal"]
    rule("NaN/inf ở cột số", ~np.isfinite(frame[num_cols].to_numpy(np.float64)).all(axis=1))
    rule("hành động ngoài [-1,1]",
         (frame["steer"].abs() > 1.0) | (frame["longitudinal"].abs() > 1.0))
    rule("previous_* ngoài [-1,1]",
         (frame["previous_steer"].abs() > 1.0) | (frame["previous_longitudinal"].abs() > 1.0))
    rule(f"|lane_offset| > {MAX_LANE_OFFSET}m", frame["lane_offset_m"].abs() > MAX_LANE_OFFSET)

    bad_heading = frame["heading_error_rad"].abs() > MAX_HEADING_ERROR
    if HEADING_RULE_SKIP_JUNCTION:
        bad_heading &= frame["is_junction"].astype(bool).eq(False)
    rule(f"|heading_error| > {MAX_HEADING_ERROR}rad"
         + (" (ngoài ngã tư)" if HEADING_RULE_SKIP_JUNCTION else ""), bad_heading)

    rule(f"speed ngoài [0, {MAX_SPEED}]",
         (frame["speed_mps"] < -1e-6) | (frame["speed_mps"] > MAX_SPEED))
    if VALID_SPEED_LIMITS:
        rule("speed_limit lạ", ~frame["speed_limit_kmh"].isin(VALID_SPEED_LIMITS))
    if CHECK_MASK_EXISTS:
        rule("file mask không tồn tại", ~frame["seg_label_path"].map(os.path.exists))

    # Frame trước bị mất -> previous_* là lệnh của rất lâu trước, tức đặc trưng SAI chứ
    # không phải thiếu, và không có gì trong dữ liệu cho biết điều đó.
    gap = frame.groupby(EPISODE_COL)["frame"].diff()
    gap_med = float(gap.median())
    rule(f"khoảng cách frame > {MAX_FRAME_GAP_MULT:.0f}x trung vị ({gap_med:.0f})",
         gap > MAX_FRAME_GAP_MULT * gap_med)
    rule(f"{SETTLE_FRAMES} frame đầu mỗi session",
         frame.groupby(EPISODE_COL).cumcount() < SETTLE_FRAMES)

    if DROP_DUPLICATE_FRAMES:
        same = ["steer", "longitudinal", "speed_mps", "lane_offset_m"]
        rule("frame trùng khít frame trước",
             frame[same].round(6).eq(frame[same].round(6).shift()).all(axis=1)
             & frame[EPISODE_COL].eq(frame[EPISODE_COL].shift()), integrity=False)

    kept = frame[~drop].reset_index(drop=True)
    rep = pd.DataFrame(report, columns=["luật", "số bản ghi"])
    rep = rep[rep["số bản ghi"] > 0]
    print(f"[{tag}] {n0} -> {len(kept)} bản ghi (bỏ {n0-len(kept)}, "
          f"{100*(n0-len(kept))/max(n0,1):.2f}%)")
    if len(rep):
        print(rep.to_string(index=False))
    return kept, rep

### 5.3 Áp dụng & báo cáo

In [ ]:
if CLEAN_ENABLED:
    _before = df.copy()
    if CLEAN_VAL_TOO:
        df, _ = clean_dataframe(df, integrity_only=True, tag="toàn bộ")
    else:
        _tr, _ = clean_dataframe(df[df["town"].isin(TRAIN_TOWNS)], True, "train")
        df = pd.concat([_tr, df[df["town"].isin(VAL_TOWNS)]], ignore_index=True)
        print("[val] không lọc (CLEAN_VAL_TOO=False)")

    # Nếu một luật vô tình cắt mất cả một HÀNH VI chứ không chỉ vài bản ghi hỏng thì nó lộ
    # ra ở đây, trước khi kịp làm hỏng cả lần train.
    _cols = ["speed_mps", "steer", "longitudinal", "lane_offset_m"]
    _cmp = pd.DataFrame({"trước": _before[_cols].abs().mean(), "sau": df[_cols].abs().mean()})
    _cmp["đổi %"] = 100 * (_cmp["sau"] / _cmp["trước"] - 1)
    print("\nTrung bình |giá trị| trước/sau:")
    print(_cmp.round(4).to_string())

    _tl_cmp = pd.DataFrame({
        "trước": _before["traffic_light_raw"].value_counts(normalize=True),
        "sau":   df["traffic_light_raw"].value_counts(normalize=True),
    }).reindex(TRAFFIC_LIGHT_RAW_VOCAB).fillna(0)
    print("\nPhân bố đèn trước/sau:")
    print(_tl_cmp.round(4).to_string())
    if (_tl_cmp["sau"] - _tl_cmp["trước"]).abs().max() > 0.05:
        print("[!] Phân bố đèn đổi hơn 5 điểm phần trăm — một luật đang cắt mất HÀNH VI, "
              "không chỉ bản ghi hỏng. Xem luật nào loại nhiều nhất ở bảng trên.")
    del _before
else:
    print("CLEAN_ENABLED = False — dùng dữ liệu thô.")
    _b2 = float((df["longitudinal"] < -0.1).mean())
    _r2 = float((df["lane_offset_m"].abs() > RECOVERY_OFFSET_THRESH).mean())
    print(f"\nSau lọc: {len(df):,} frame | phanh {100*_b2:.1f}% (thô {100*_brake:.1f}%) "
          f"| lệch >{RECOVERY_OFFSET_THRESH}m {100*_r2:.1f}% (thô {100*_recov:.1f}%)")
    if abs(_b2 - _brake) > 0.05 or abs(_r2 - _recov) > 0.05:
        print("[!] Một trong hai tỉ lệ đổi hơn 5 điểm phần trăm — luật lọc đang cắt HÀNH VI.")


### 3b. (Tuỳ chọn) Dùng segmentation dự đoán thay ground-truth

Chỉ chạy khi `USE_PREDICTED_SEGMENTATION = True`. Suy luận ở đúng độ phân giải seg được
train rồi mới hạ xuống kích thước IL bằng `downscale_labels` — cùng đường đi với
ground-truth ở §8.

In [ ]:
def _strip_scse_bias(module):
    """Bỏ bias của 3 conv 1x1 trong mọi SCSEModule — bản sao `align_scse_` của train-seg.

    train-seg.ipynb §8 phải vá như vậy để tránh "CUDA error: misaligned address" của
    DataParallel (bias numel 1 và 2 làm lệch địa chỉ mọi tensor phía sau trong buffer
    broadcast). Checkpoint vì thế KHÔNG chứa các key bias đó. Dựng smp.Unet mặc định rồi
    load_state_dict sẽ nổ "Missing key(s): decoder.blocks.0.attention1.cSE.1.bias, ...".
    Phải vá y hệt TRƯỚC khi nạp trọng số.
    """
    def no_bias(conv):
        new = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size,
                        stride=conv.stride, padding=conv.padding,
                        dilation=conv.dilation, groups=conv.groups, bias=False)
        with torch.no_grad():
            new.weight.copy_(conv.weight)
        return new

    n = 0
    for m in module.modules():
        if type(m).__name__ == "SCSEModule":
            m.cSE[1] = no_bias(m.cSE[1])
            m.cSE[3] = no_bias(m.cSE[3])
            m.sSE[0] = no_bias(m.sSE[0])
            n += 1
    return n


if USE_PREDICTED_SEGMENTATION:
    os.makedirs(PREDICTED_MASK_DIR, exist_ok=True)
    _ck = torch.load(SEG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)

    # Đọc kiến trúc THẲNG từ checkpoint thay vì chép tay: dựng sai lớp thì load_state_dict
    # báo thiếu/thừa key. `decoder_attention_type` phải lấy từ file — hardcode "scse" ở đây
    # sẽ hỏng ngay khi seg được train lại với DECODER_ATTENTION = None.
    _arch = _ck.get("model_arch", "unet")
    _enc  = _ck.get("encoder_name", "resnet34")
    _att  = _ck.get("decoder_attention_type", "scse")
    if _arch == "unet":
        seg_model = smp.Unet(encoder_name=_enc, encoder_weights=None,
                             decoder_attention_type=_att, in_channels=3,
                             classes=_ck["num_classes"])
    else:
        seg_model = smp.DeepLabV3Plus(encoder_name=_enc, encoder_weights=None,
                                      encoder_output_stride=8, in_channels=3,
                                      classes=_ck["num_classes"])

    if _ck.get("scse_bias_free"):
        print(f"vá SCSE bias-free: {_strip_scse_bias(seg_model)} module (khớp checkpoint seg)")

    seg_model.load_state_dict(_ck["model_state_dict"])   # strict=True: sai kiến trúc là nổ ngay
    seg_model.to(DEVICE).eval().requires_grad_(False)

    # Đây là bước DUY NHẤT trong notebook đáng dùng cả 2 GPU: 40k ảnh x 2 forward (flip-TTA)
    # bằng ~8 epoch val của notebook segmentation. Suy luận thuần nên DataParallel gần như
    # scale tuyến tính (không có gradient để gom về GPU0).
    seg_net = nn.DataParallel(seg_model) if N_GPU > 1 else seg_model
    SEG_INFER_BATCH = 32 * max(N_GPU, 1)

    _H = _ck.get("image_height", SEG_NATIVE_HEIGHT)
    _W = _ck.get("image_width",  SEG_NATIVE_WIDTH)
    infer_tf = A.Compose([
        A.Resize(height=_H, width=_W, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=tuple(_ck.get("norm_mean", (0.485, 0.456, 0.406))),
                    std=tuple(_ck.get("norm_std",  (0.229, 0.224, 0.225)))),
        ToTensorV2(),
    ])
    print(f"seg: {_arch}/{_enc} attention={_att} @ {_H}x{_W}, {_ck['num_classes']} lớp "
          f"{_ck['class_names']} | batch {SEG_INFER_BATCH} trên {max(N_GPU, 1)} GPU")

    # Khoá cache là "session_id + tên file", KHÔNG phải riêng basename: "frame" đếm lại
    # từ đầu mỗi phiên nên các Town có dải trùng nhau.
    _rgb_paths = df["rgb_path"].to_numpy()
    predicted_paths = [
        os.path.join(PREDICTED_MASK_DIR,
                     f"{s}_{os.path.basename(p)}".replace(".png", "_pred.png"))
        for s, p in zip(df[EPISODE_COL], _rgb_paths)]
    todo = [i for i, p in enumerate(predicted_paths) if not os.path.exists(p)]

    class _RgbInferDataset(Dataset):
        """Đọc + chuẩn hoá ảnh trong DataLoader worker, không chặn GPU như vòng lặp bs=1."""

        def __init__(self, indices):
            self.indices = indices

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, k):
            i = self.indices[k]
            img = cv2.imread(_rgb_paths[i], cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(_rgb_paths[i])
            return infer_tf(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))["image"], i

    if todo:
        # Workers ở đây là CẦN: mỗi mẫu phải decode 1 JPEG/PNG 384x480 + normalize, khác hẳn
        # DataLoader train (§8) vốn chỉ index vào cache trong RAM.
        infer_loader = DataLoader(_RgbInferDataset(todo), batch_size=SEG_INFER_BATCH,
                                  shuffle=False, num_workers=SEG_INFER_WORKERS,
                                  pin_memory=torch.cuda.is_available())
        with torch.no_grad():
            for _imgs, _idx in tqdm(infer_loader, desc="Sinh segmentation dự đoán"):
                _imgs = _imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    _logits = seg_net(_imgs) + torch.flip(
                        seg_net(torch.flip(_imgs, dims=[3])), dims=[3])
                _pred = _logits.argmax(1).to(torch.uint8).cpu().numpy()
                for _j, _i in enumerate(_idx.tolist()):
                    cv2.imwrite(predicted_paths[_i], _pred[_j])
    else:
        print("Tất cả mask dự đoán đã có sẵn — bỏ qua bước suy luận.")

    df["seg_label_path"] = predicted_paths
    SEG_PATHS_ARE_TRAIN_IDS = True
    del seg_model, seg_net, _ck
    torch.cuda.empty_cache()
    print(f"Đã chuyển sang segmentation dự đoán (train id 0..{NUM_CLASSES - 1}) tại "
          f"{PREDICTED_MASK_DIR}.")
else:
    print("Dùng ground-truth segmentation (raw id CARLA -> LUT).")
    print("Lưu ý: DRL lúc chạy thật chỉ có mask DỰ ĐOÁN, nơi Sidewalk IoU = 0.81 và "
          "RoadLine = 0.89.\n  Đặt USE_PREDICTED_SEGMENTATION = True ở §2 để khớp đúng "
          "phân phối đó (+~20-30 phút).")

## 6. Chia train/val theo town

### 6.1 Chia & lọc dư thừa (chỉ train)

In [ ]:
train_df = df[df["town"].isin(TRAIN_TOWNS)].reset_index(drop=True)
val_df   = df[df["town"].isin(VAL_TOWNS)].reset_index(drop=True)
assert len(train_df) and len(val_df), \
    f"Chia theo town thất bại — town có mặt: {sorted(df['town'].unique())}"

print(f"Chia theo TOWN: train={TRAIN_TOWNS} val={VAL_TOWNS}")
_counts = df.groupby("town").size()
_median = float(_counts.median())
for _t, _n in _counts.items():
    _dev = (_n - _median) / max(_median, 1)
    print(f"  {_t}: {_n} mẫu ({_dev:+.0%} so với trung vị {_median:.0f})"
          + ("   <-- lệch, cân nhắc thu bù" if abs(_dev) > TOWN_IMBALANCE_TOL else ""))

# --- Lọc chuỗi đứng yên: CHỈ trên train --------------------------------------------------
# val KHÔNG bao giờ bị lọc/cân bằng lại. Đó là điều kiện để mọi con số §13 so sánh được
# giữa các lần chạy: chỉ có model thay đổi, thước đo thì đứng yên.
if CLEAN_ENABLED:
    # Luật dư thừa (frame trùng khít) — CHỈ train, val giữ nguyên làm thước đo.
    train_df, _ = clean_dataframe(train_df, integrity_only=False, tag="train/dư thừa")

if DROP_STATIONARY_RUNS:
    _still = train_df["speed_mps"].abs() < STATIONARY_SPEED
    # Đánh số từng chuỗi đứng yên liên tiếp trong mỗi session, rồi giữ STATIONARY_KEEP
    # frame đầu của chuỗi — đủ để model học "thấy đèn đỏ thì dừng và GIỮ dừng".
    _grp = (_still != _still.shift()).cumsum()
    _rank = train_df.groupby([EPISODE_COL, _grp]).cumcount()
    _keep = (~_still) | (_rank < STATIONARY_KEEP)
    print(f"\nLọc chuỗi đứng yên (train): bỏ {int((~_keep).sum())} / {len(train_df)} mẫu "
          f"({100*float((~_keep).mean()):.1f}%), giữ tối đa {STATIONARY_KEEP} frame/chuỗi")
    train_df = train_df[_keep].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} mẫu ({train_df[EPISODE_COL].nunique()} session) | "
      f"Val: {len(val_df)} mẫu ({val_df[EPISODE_COL].nunique()} session)")
print("\nPhân bố đèn tín hiệu (train / val):")
print(pd.DataFrame({
    "train": train_df["traffic_light_state"].value_counts(normalize=True),
    "val":   val_df["traffic_light_state"].value_counts(normalize=True),
}).reindex(TRAFFIC_LIGHT_VOCAB).fillna(0).round(4).to_string())
if not (val_df["traffic_light_state"] == "green").any():
    print("[!] Val KHÔNG có mẫu đèn xanh nào -> dòng 'green' ở bảng MAE §13 sẽ là NaN.")

### 6.2 Lệch phân phối train/val & dữ liệu recovery

In [ ]:
# --- Lệch phân phối train/val: cái bẫy tự tạo ra bằng chính các cờ lọc ------------------
# Chia theo town vốn đã tạo domain shift (đó là chủ ý). Nhưng nếu một cờ lọc làm lệch THÊM
# về HÀNH VI thì model học một bài toán rồi bị chấm bằng bài khác. Tổng biến sai khác
# (total variation) = một nửa tổng chênh lệch tuyệt đối giữa hai phân phối.
_tv = 0.5 * (train_df["traffic_light_state"].value_counts(normalize=True)
             .reindex(TRAFFIC_LIGHT_VOCAB).fillna(0)
             - val_df["traffic_light_state"].value_counts(normalize=True)
             .reindex(TRAFFIC_LIGHT_VOCAB).fillna(0)).abs().sum()
_brake_tr = float((train_df["longitudinal"] < -0.1).mean())
_brake_va = float((val_df["longitudinal"] < -0.1).mean())
print(f"\nLệch phân phối train/val: TV(đèn) = {_tv:.3f} | tỉ lệ phanh "
      f"train {100*_brake_tr:.1f}% vs val {100*_brake_va:.1f}%")
if _tv > 0.25 or abs(_brake_tr - _brake_va) > 0.15:
    print("[!] LỆCH LỚN. Model sẽ học một phân phối rồi bị chấm trên một phân phối khác —")
    print("    đây chính là cách longitudinal MAE nhảy 0.0221 -> 0.1919 ở lần chạy v4.")
    print("    Nghi phạm số một: DROP_STATIONARY_RUNS (bỏ frame dừng đèn đỏ = bỏ cả pha")
    print("    phanh). Tắt nó, hoặc tăng mạnh STATIONARY_KEEP.")

# --- Dữ liệu hồi phục: có bao nhiêu frame ĐÃ lệch làn để học cách quay về? --------------
for _name, _part in (("train", train_df), ("val", val_df)):
    _off = _part["lane_offset_m"].abs()
    _share = float((_off > RECOVERY_OFFSET_THRESH).mean())
    print(f"[{_name}] {100*_share:5.1f}% mẫu lệch >{RECOVERY_OFFSET_THRESH}m "
          f"({int((_off > RECOVERY_OFFSET_THRESH).sum())} frame) | "
          f"p95 |offset| = {_off.quantile(.95):.3f}m | max = {_off.max():.3f}m")
if float((train_df["lane_offset_m"].abs() > RECOVERY_OFFSET_THRESH).mean()) < 0.05:
    print("[!] Dưới 5% dữ liệu train nằm ngoài tâm làn. Model sẽ giỏi khi đang đi đúng và")
    print("    lạc lối ngay khi trôi — trong vòng kín đó là sai số TỰ KHUẾCH ĐẠI.")
    print("    Hai lối ra: SAMPLER_MODE = 'offset' (dùng lại dữ liệu sẵn có), hoặc thu thêm")
    print("    pha recovery (đặt xe lệch 0.3-0.8m rồi ghi lại lúc autopilot lái về).")

### 6.3 Baseline — ngưỡng phải vượt ở §13

In [ ]:
# --- BA baseline. Cả ba đều đo trên val, không học gì cả ---------------------------------
# 1. "chép"      : steer_t = previous_steer. Trả lời "model có học được gì từ ẢNH không, hay
#                  chỉ phát lại đầu vào?". Đây là ngưỡng QUYẾT ĐỊNH checkpoint có dùng được.
# 2. "hằng số"   : steer = 0 / longitudinal = trung bình train. Trả lời "model có học được
#                  gì KHÔNG?". Là mốc dễ — vượt nó không phải thành tựu, nhưng KHÔNG vượt
#                  nổi thì model vô dụng hoàn toàn.
# Hằng số cho longitudinal lấy từ TRAIN (không phải val) — cùng nguyên tắc chống rò rỉ như
# norm_stats ở §7; dùng trung bình val là tự cho baseline xem trước đáp án.
COPYCAT_STEER_MAE = float((val_df["steer"] - val_df["previous_steer"]).abs().mean())
COPYCAT_LONG_MAE  = float((val_df["longitudinal"] - val_df["previous_longitudinal"]).abs().mean())
CONST_STEER_MAE   = float(val_df["steer"].abs().mean())
_long_const       = float(train_df["longitudinal"].mean())
CONST_LONG_MAE    = float((val_df["longitudinal"] - _long_const).abs().mean())

for name, part in (("train", train_df), ("val", val_df)):
    r = part["steer"].corr(part["previous_steer"])
    mae = (part["steer"] - part["previous_steer"]).abs().mean()
    print(f"\n[{name}] corr(steer, previous_steer) = {r:.4f}"
          f"   MAE baseline 'chép' = {mae:.4f}")

print(f"\n{'':<16}{'chép prev':>12}{'hằng số':>12}")
print(f"{'Steer':<16}{COPYCAT_STEER_MAE:>12.4f}{CONST_STEER_MAE:>12.4f}")
print(f"{'Longitudinal':<16}{COPYCAT_LONG_MAE:>12.4f}{CONST_LONG_MAE:>12.4f}   "
      f"(hằng số = {_long_const:.3f})")
print("\n§13 phải THẮNG cột 'chép prev'. Cột 'hằng số' chỉ để thấy độ khó cơ bản của bài "
      "toán\nvà để báo cáo không bị đọc thành 'model không học được gì'.")
if val_df["steer"].corr(val_df["previous_steer"]) > 0.98:
    print("[!] corr > 0.98: bước thời gian quá dày, baseline 'chép' gần như bất khả chiến "
          "bại.\n    Lấy thưa dữ liệu hoặc đặt USE_PREV_ACTIONS = False ở §2.")

## 7. Chuẩn hoá đặc trưng số

In [ ]:
norm_stats = {c: (float(train_df[c].mean()), float(train_df[c].std()) + 1e-6)
              for c in CONTINUOUS_COLS}
NORM_MEAN = np.array([norm_stats[c][0] for c in CONTINUOUS_COLS], dtype=np.float32)
NORM_STD  = np.array([norm_stats[c][1] for c in CONTINUOUS_COLS], dtype=np.float32)
for c, (m, s) in norm_stats.items():
    print(f"{c}: mean={m:.4f} std={s:.4f}")

## 8. Dataset & DataLoader

### 8.1 Đọc mask + lớp Dataset

In [ ]:
def load_mask(path):
    """PNG -> train id ở kích thước IL. THỨ TỰ BẮT BUỘC: remap trước, hạ mẫu sau.

    Resize trước rồi mới remap thì vạch kẻ đã bị NEAREST xoá từ lúc còn là raw id.
    """
    # IMREAD_UNCHANGED chứ không phải IMREAD_GRAYSCALE — giống `read_mask_raw` của
    # train-seg.ipynb. Collector ghi seg_label 1 kênh, nhưng nếu một phiên nào đó ghi ra
    # ảnh 3 kênh thì tag nằm ở kênh đỏ; GRAYSCALE sẽ trộn ba kênh theo trọng số độ sáng và
    # sinh ra raw id KHÔNG TỒN TẠI — mà mọi id lạ đều lặng lẽ rơi về Background qua LUT.
    mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise FileNotFoundError(path)
    if mask.ndim == 3:
        mask = np.ascontiguousarray(mask[:, :, 2])
    if not SEG_PATHS_ARE_TRAIN_IDS:
        mask = SEG_LABEL_LUT[mask]
    mask = downscale_labels(mask)
    if mask.max() >= NUM_CLASSES:
        raise ValueError(f"train id {mask.max()} >= NUM_CLASSES={NUM_CLASSES} tại {path} "
                         "— LUT hoặc checkpoint seg không khớp.")
    return mask


class SteeringDataset(Dataset):
    def __init__(self, frame, augment, cache=CACHE_MASKS_IN_RAM):
        frame = frame.reset_index(drop=True)
        self.augment = augment
        self.use_prev = len(RAW_ACTION_COLS) > 0
        # Rút sẵn thành mảng numpy: df.iloc[idx] tốn ~50 us/mẫu, nhân với vài triệu lượt đọc.
        self.paths  = frame["seg_label_path"].to_numpy()
        self.cont   = frame[CONTINUOUS_COLS].to_numpy(np.float32)   # RAW, chuẩn hoá sau khi lật
        self.prev   = frame[RAW_ACTION_COLS].to_numpy(np.float32)   # (N, 0) nếu tắt prev
        self.target = frame[["steer", "longitudinal"]].to_numpy(np.float32)
        self.aux    = frame[AUX_COLS].to_numpy(np.float32)
        self.tl = np.stack([(frame["traffic_light_state"] == v).to_numpy(np.float32)
                            for v in TRAFFIC_LIGHT_VOCAB], axis=1)
        # v9 đã bỏ yaw_rate_rps khỏi CONTINUOUS_COLS (§2.3), nên cột này có thể không
        # tồn tại. Giữ nhánh xử lý để vẫn chạy được nếu ai đó bật lại nó để đối chứng.
        self.yaw_col = (CONTINUOUS_COLS.index("yaw_rate_rps")
                        if "yaw_rate_rps" in CONTINUOUS_COLS else None)

        self.masks = None
        if cache:
            # Đọc + remap + hạ mẫu 40k PNG mất ~6 phút nếu chạy tuần tự, và nó nằm TRƯỚC
            # mọi epoch nên là phần thời gian chết lớn nhất của notebook. cv2.imread/resize
            # đều nhả GIL -> ThreadPoolExecutor ăn đủ 4 vCPU của Kaggle mà không phải pickle
            # gì (khác multiprocessing). pool.map giữ THỨ TỰ nên i vẫn khớp self.paths.
            self.masks = np.empty((len(frame), IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
            n_thread = max(os.cpu_count() or 2, 1)
            with ThreadPoolExecutor(max_workers=n_thread) as pool:
                for i, m in enumerate(tqdm(pool.map(load_mask, self.paths),
                                           total=len(self.paths),
                                           desc="cache mask", leave=False)):
                    self.masks[i] = m
            print(f"  cache mask: {self.masks.nbytes/1e9:.2f} GB cho {len(frame)} frame "
                  f"({n_thread} thread)")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        mask = self.masks[i] if self.masks is not None else load_mask(self.paths[i])
        cont, prev, aux = self.cont[i].copy(), self.prev[i].copy(), self.aux[i].copy()
        steer, longitudinal = self.target[i]

        if self.augment and random.random() < 0.5:
            mask = mask[:, ::-1]
            steer = -steer
            if self.yaw_col is not None:
                cont[self.yaw_col] = -cont[self.yaw_col]
            aux[0], aux[1] = -aux[0], -aux[1]
            if self.use_prev:                 # prev rỗng khi USE_PREV_ACTIONS = False
                prev[0] = -prev[0]

        scalar = np.concatenate([(cont - NORM_MEAN) / NORM_STD, prev, self.tl[i]])
        return (torch.from_numpy(np.ascontiguousarray(mask)),
                torch.from_numpy(scalar.astype(np.float32)),
                torch.from_numpy(aux),
                torch.tensor([steer, longitudinal]))

### 8.2 Dựng dataset & cache mask vào RAM

In [ ]:
train_ds = SteeringDataset(train_df, augment=True)
val_ds   = SteeringDataset(val_df, augment=False)
assert train_ds[0][1].shape[0] == SCALAR_FEATURE_DIM, (
    f"scalar vector dài {train_ds[0][1].shape[0]} nhưng SCALAR_FEATURE_DIM="
    f"{SCALAR_FEATURE_DIM} — checkpoint sẽ ghi sai hợp đồng cho DRL.")

if CACHE_MASKS_IN_RAM and os.name == "nt" and NUM_WORKERS > 0:
    NUM_WORKERS = 0
    print("[!] Windows + cache mask -> NUM_WORKERS = 0 (tránh nhân bản cache theo worker)")

# persistent_workers CHỈ bật cho train_loader. Dùng chung một dict cho cả hai loader là đúng
# cái làm train-seg rò +2.9 GB/epoch: worker của train không bao giờ chết trong khi worker
# của val được tạo thêm mỗi epoch. Với NUM_WORKERS = 0 (mặc định khi cache mask) đây là no-op.
_common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
               generator=DATALOADER_GENERATOR,
               worker_init_fn=seed_worker if NUM_WORKERS > 0 else None)
if NUM_WORKERS > 0:
    _common["prefetch_factor"] = 4
_train_kwargs = dict(_common, persistent_workers=NUM_WORKERS > 0)
_val_kwargs   = dict(_common, persistent_workers=False)

### 8.3 Sampler & DataLoader

In [ ]:
def _class_weights(labels, power=None, cap=None):
    """w = (1/tần_suất) ** power, chuẩn hoá về trung bình 1, rồi cắt trần.

    Chuẩn hoá để `cap` có nghĩa tuyệt đối: sau bước này 1.0 = "lấy mẫu như bình thường",
    cap = "nhiều nhất bấy nhiêu lần bình thường". `power` mới là knob chính — nghịch đảo
    tần suất đầy đủ (power=1) đẩy tỉ số straight/sharp lên 23 lần, quá mạnh cho một tập chỉ
    có 3.5% mẫu cua gấp.
    """
    power = SAMPLER_POWER if power is None else power
    cap = SAMPLER_WEIGHT_CAP if cap is None else cap
    freq = labels.value_counts(normalize=True)
    w = (1.0 / labels.astype(object).map(freq).to_numpy(np.float64)) ** power
    w /= w.mean()
    return np.clip(w, None, cap)


steer_bins = pd.cut(train_df["steer"].abs(), bins=[-0.001, 0.02, 0.1, 0.3, np.inf],
                    labels=["straight", "gentle", "moderate", "sharp"])
print("Phân bố bin |steer| (train):")
print(steer_bins.value_counts(normalize=True).round(3).to_string())

offset_bins = pd.cut(train_df["lane_offset_m"].abs(), bins=OFFSET_BIN_EDGES,
                     labels=OFFSET_BIN_LABELS)
print("Phân bố bin |lane_offset_m| (train):")
print(offset_bins.value_counts(normalize=True).reindex(OFFSET_BIN_LABELS).round(4).to_string())

if SAMPLER_MODE == "none":
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, **_train_kwargs)
    print("Sampler: KHÔNG cân bằng — phân phối train giống val, cơ hội cao nhất vượt "
          "baseline ở §13.")
else:
    _parts = SAMPLER_MODE.split("+")
    _known = {"steer", "tl", "offset"}
    if not set(_parts) <= _known:
        raise ValueError(f"SAMPLER_MODE lạ: {SAMPLER_MODE} (hợp lệ: {sorted(_known)})")
    w = np.ones(len(train_df), dtype=np.float64)
    if "steer" in _parts:
        w *= _class_weights(steer_bins)
    if "offset" in _parts:
        # Đây là knob nhắm thẳng vào phát hiện của v3: 13.8% mẫu lệch làn gây 64% sai số.
        w *= _class_weights(offset_bins)
    if "tl" in _parts:
        w *= _class_weights(train_df["traffic_light_state"])
        print("[!] 'tl' coi 'red' là hiếm, trong khi frame dừng đèn đỏ lại là nhóm TRÙNG")
        print("    LẶP nhất (steer=0, brake=-1). Cân nhắc DROP_STATIONARY_RUNS=True.")
    w = np.clip(w / w.mean(), None, SAMPLER_WEIGHT_CAP)

    sampler = WeightedRandomSampler(w, num_samples=len(train_df), replacement=True,
                                    generator=DATALOADER_GENERATOR)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              drop_last=True, **_train_kwargs)
    print(f"Sampler '{SAMPLER_MODE}' (power={SAMPLER_POWER}): trọng số "
          f"[{w.min():.2f}, {w.max():.2f}], tỉ số {w.max()/max(w.min(),1e-9):.1f}x "
          f"(trần {SAMPLER_WEIGHT_CAP}). Lưu ý: phân phối train giờ KHÁC val, nên MAE "
          f"không trọng số ở §13 sẽ xấu đi — đó là đánh đổi có chủ đích.")

# shuffle=False + drop_last=False: §13 dựa vào việc thứ tự batch val trùng thứ tự val_df.
val_loader = DataLoader(val_ds, batch_size=max(BATCH_SIZE, 256), shuffle=False, **_val_kwargs)
print(f"train {len(train_ds)} mẫu / {len(train_loader)} batch | "
      f"val {len(val_ds)} mẫu / {len(val_loader)} batch")

## 9. Trực quan hoá mask sau tiền xử lý

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    lab, scalar, aux, target = train_ds[random.randrange(len(train_ds))]
    lab = lab.numpy()
    axes[i].imshow(PALETTE[lab])
    # % RoadLine để kiểm tra bằng mắt rằng vạch kẻ sống sót qua bước hạ mẫu 384->192.
    axes[i].set_title(f"steer={target[0]:.2f} long={target[1]:.2f} | "
                      f"RoadLine {100*(lab==ROADLINE_ID).mean():.2f}%")
    axes[i].axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, fc=PALETTE[i] / 255) for i in range(NUM_CLASSES)]
fig.legend(handles, CLASS_NAMES, ncol=NUM_CLASSES, loc="lower center", frameon=False)
plt.tight_layout()
plt.savefig("bieu_do_mask.png", dpi=300, bbox_inches="tight")
plt.show()
print("RoadLine ở ground-truth gốc chiếm ~1.06% pixel. Xuống dưới ~0.6% nghĩa là "
      "downscale_labels chưa chạy đúng — vạch kẻ đang bị bước hạ mẫu ăn mất.")

## 10. Model

### 10.1 Kiến trúc SteeringNet

In [ ]:
import copy, math

# Thang đo của hai chiều hành động, tính TRÊN TRAIN (cùng nguyên tắc chống rò rỉ như
# norm_stats ở §7). Dùng để cân bằng gradient giữa steer và longitudinal — xem `control_loss`.
ACTION_STD = np.array([
    float(train_df["steer"].std()) + 1e-6,
    float(train_df["longitudinal"].std()) + 1e-6,
], dtype=np.float32)
AUX_TARGET_IDX = [AUX_COLS.index(c) for c in AUX_TARGET_COLS]
AUX_SCALE_T  = torch.tensor(AUX_TARGET_SCALE, dtype=torch.float32, device=DEVICE)
ACTION_STD_T = torch.tensor(ACTION_STD, device=DEVICE)

print(f"std hành động trên train: steer {ACTION_STD[0]:.4f} | long {ACTION_STD[1]:.4f} "
      f"(tỉ số {ACTION_STD[1]/ACTION_STD[0]:.1f}x)")
print(f"mục tiêu phụ: {AUX_TARGET_COLS} (cột aux {AUX_TARGET_IDX}), trọng số {AUX_LOSS_WEIGHT}")


class SteeringNet(nn.Module):
    """CNN (segmentation) + MLP (scalar) -> [steer, longitudinal], kèm ĐẦU RA PHỤ.

    Kiến trúc và TÊN thuộc tính (`conv`, `pool`, `cnn_fc`, `scalar_mlp`, `head`) phải khớp
    `drl_training/policy/backbone.py` + `actor_critic.py` — đó là nơi actor PPO nạp lại
    trọng số warm-start từ checkpoint này.

    Hai thay đổi của v9 so với v7, cả hai đều đã đồng bộ sẵn sang `backbone.py`:

    1. `pool` là AdaptiveAvgPool2d(POOL_GRID) chứ không còn (1, 1). Trung bình toàn ảnh bóp
       bản đồ đặc trưng 12x15 thành một số mỗi kênh; lưới 4x6 giữ lại bố cục gần-xa và
       trái-phải, tức thứ mà bài toán bám làn thực sự cần.

    2. `aux_head` — nhánh MỚI, chỉ có ở IL, KHÔNG mang sang DRL. Nó cắm thẳng vào `x_img`
       (đặc trưng ảnh, TRƯỚC khi ghép với scalar) và phải dự đoán `lane_offset_m` +
       `heading_error_rad`. Vì nó không thấy scalar, cách duy nhất để giảm loss phụ là mã
       hoá vị trí ngang vào chính đặc trưng ảnh — đúng thứ v7 chưa bao giờ học.
       `drl_training/policy/il_compat.py` lọc theo tiền tố (`conv.`/`pool.`/`cnn_fc.`/
       `scalar_mlp.` và đúng `head.0`/`head.3`/`head.6`) nên `aux_head.*` bị bỏ qua khi
       warm-start, không cần làm gì thêm bên DRL.
    """

    def __init__(self, num_classes=NUM_CLASSES, num_scalar_features=SCALAR_FEATURE_DIM,
                 pool_grid=POOL_GRID, num_aux=len(AUX_TARGET_COLS)):
        super().__init__()
        self.num_classes = num_classes
        self.conv = nn.Sequential(
            nn.Conv2d(num_classes, 24, 5, 2, 2), nn.BatchNorm2d(24), nn.ELU(),
            nn.Conv2d(24, 36, 5, 2, 2), nn.BatchNorm2d(36), nn.ELU(),
            nn.Conv2d(36, 48, 5, 2, 2), nn.BatchNorm2d(48), nn.ELU(),
            nn.Conv2d(48, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(pool_grid)
        self.cnn_fc = nn.Sequential(nn.Linear(64 * pool_grid[0] * pool_grid[1], 64), nn.ELU())
        self.scalar_mlp = nn.Sequential(nn.Linear(num_scalar_features, 32), nn.ELU(),
                                        nn.Linear(32, 32), nn.ELU())
        self.head = nn.Sequential(
            nn.Linear(64 + 32, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(0.2),
            nn.Linear(32, 2),
        )
        # Không Dropout ở nhánh phụ: nó là công cụ định hình đặc trưng, không phải đầu ra
        # cuối cùng, và nhiễu ở đây chỉ làm loãng chính tín hiệu ta đang cố tạo ra.
        self.aux_head = nn.Sequential(nn.Linear(64, 32), nn.ELU(), nn.Linear(32, num_aux))

    def image_features(self, seg_map):
        if seg_map.dim() == 3:
            seg_map = F.one_hot(seg_map.long(), self.num_classes).permute(0, 3, 1, 2).float()
        return self.cnn_fc(self.pool(self.conv(seg_map)).flatten(1))

    def forward(self, seg_map, scalar_features, return_aux=False):
        """seg_map: (N, H, W) class-id map, hoặc (N, num_classes, H, W) one-hot float."""
        x_img = self.image_features(seg_map)
        x_sca = self.scalar_mlp(scalar_features)
        action = torch.tanh(self.head(torch.cat([x_img, x_sca], dim=1)))
        if return_aux:
            return action, self.aux_head(x_img)
        return action

### 10.2 Loss, điểm chọn checkpoint, EMA

In [ ]:
def control_loss(pred, target, pred_aux=None, target_aux=None):
    """SmoothL1 với HUBER_BETA nhỏ ~ MAE có làm mượt quanh 0, cộng loss phụ.

    Vì sao không dùng thẳng MSE: metric ở §13 là MAE, mà MSE cho gradient tỉ lệ thuận với
    sai số nên các mẫu "đi thẳng" (80% dữ liệu, sai số ~0.005) gần như không tạo áp lực học.
    Vì sao không dùng thẳng L1: đạo hàm gián đoạn tại 0 làm nghiệm dao động quanh đáy khi LR
    còn cao. Huber beta nhỏ lấy phần tốt của cả hai.

    Vì sao chia cho ACTION_STD (v9): |steer| điển hình 0.005-0.03 còn |longitudinal| chạy cả
    dải [-1, 1]. Dùng chung một `beta` nghĩa là chiều longitudinal áp đảo gradient, và
    "xuất steer ≈ 0" trở thành cực tiểu rẻ nhất — chính xác thứ v4 rơi vào. Chia mỗi chiều
    cho độ lệch chuẩn của nó đưa cả hai về cùng thang, và `beta` cũng thành tương đối.
    Không đổi không gian ĐẦU RA (model vẫn xuất steer thật, đã tanh) nên `drl_training`
    không phải sửa gì.
    """
    if NORMALIZE_ACTION_LOSS:
        err = (pred - target) / ACTION_STD_T
        steer = F.smooth_l1_loss(err[:, 0], torch.zeros_like(err[:, 0]), beta=HUBER_BETA)
        longitudinal = F.smooth_l1_loss(err[:, 1], torch.zeros_like(err[:, 1]), beta=HUBER_BETA)
    else:
        steer = F.smooth_l1_loss(pred[:, 0], target[:, 0], beta=HUBER_BETA)
        longitudinal = F.smooth_l1_loss(pred[:, 1], target[:, 1], beta=HUBER_BETA)
    loss = STEER_LOSS_WEIGHT * steer + LONGITUDINAL_LOSS_WEIGHT * longitudinal

    if pred_aux is not None and AUX_LOSS_WEIGHT > 0:
        loss = loss + AUX_LOSS_WEIGHT * F.smooth_l1_loss(
            pred_aux / AUX_SCALE_T, target_aux / AUX_SCALE_T, beta=0.1)
    return loss


def val_score(steer_mae, long_mae):
    """Điểm CHỌN CHECKPOINT. Cùng trọng số với loss nhưng tính trên MAE.

    Bản cũ chọn theo val_loss — mà val_loss là Huber còn thước đo cuối là MAE, nên checkpoint
    được giữ lại chưa chắc là checkpoint tốt nhất theo tiêu chí thật sự dùng để chấm.

    v9: cũng chuẩn hoá theo ACTION_STD như `control_loss`, nếu không thì điểm CHỌN checkpoint
    và hàm đang TỐI ƯU lại nói hai ngôn ngữ khác nhau.
    """
    if NORMALIZE_ACTION_LOSS:
        steer_mae = steer_mae / float(ACTION_STD[0])
        long_mae = long_mae / float(ACTION_STD[1])
    return STEER_LOSS_WEIGHT * steer_mae + LONGITUDINAL_LOSS_WEIGHT * long_mae


class ModelEMA:
    """Trung bình trượt trọng số. decay tăng dần để những step đầu không bị khởi tạo kéo lại."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval().requires_grad_(False)
        self.decay, self.step = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for e, m in zip(self.module.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(m.detach(), alpha=1.0 - d)
            else:
                e.copy_(m)          # num_batches_tracked của BatchNorm là int

### 10.3 Optimizer & lịch học

In [ ]:
# SteeringNet chỉ ~0.15M tham số: DataParallel không được dùng vì chi phí đồng bộ giữa 2 GPU
# lớn hơn phần tính toán tiết kiệm được, và nó thêm tiền tố "module." vào state_dict.
model = SteeringNet().to(DEVICE)
ema = ModelEMA(model, EMA_DECAY) if USE_EMA else None

# Bias/BatchNorm không chịu weight decay (thực hành chuẩn của AdamW).
decay, no_decay = [], []
for _n, _p in model.named_parameters():
    if _p.requires_grad:
        (no_decay if _p.ndim <= 1 or _n.endswith(".bias") else decay).append(_p)
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                               {"params": no_decay, "weight_decay": 0.0}], lr=LEARNING_RATE)

# Lịch theo STEP, không theo epoch: warmup 200 step rồi cosine xuống MIN_LR_RATIO. Warmup
# cần vì LR đã được scale lên 2e-3 theo batch 128 — vài chục step đầu ở LR đó đủ để đẩy
# BatchNorm vào trạng thái xấu.
_TOTAL_STEPS = max(EPOCHS * len(train_loader), 1)


def _lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    p = min((step - WARMUP_STEPS) / max(1, _TOTAL_STEPS - WARMUP_STEPS), 1.0)
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
_n_img = sum(p.numel() for n, p in model.named_parameters()
             if n.startswith(("conv.", "cnn_fc.")))
print(f"Số tham số: {sum(p.numel() for p in model.parameters())/1e6:.3f}M "
      f"(nhánh ảnh {_n_img/1e6:.3f}M) | pool {POOL_GRID} "
      f"| scalar_dim={SCALAR_FEATURE_DIM} {SCALAR_FEATURE_ORDER}")
print(f"{_TOTAL_STEPS:,} step | warmup {WARMUP_STEPS} | lr {LEARNING_RATE:.2e} -> "
      f"{LEARNING_RATE*MIN_LR_RATIO:.2e} | huber_beta={HUBER_BETA} | ema={USE_EMA}")
print(f"chuẩn hoá loss theo std hành động: {NORMALIZE_ACTION_LOSS}")

# Chốt chặn khớp kiến trúc với DRL: shape của `cnn_fc.0.weight` phụ thuộc POOL_GRID, và nếu
# hai bên lệch thì lỗi chỉ lộ ra lúc warm-start PPO, sau khi đã train xong cả tiếng.
assert model.cnn_fc[0].in_features == 64 * POOL_GRID[0] * POOL_GRID[1]
print(f"cnn_fc nhận {model.cnn_fc[0].in_features} chiều — drl_training/policy/backbone.py "
      f"phải có POOL_GRID = {POOL_GRID}")

## 11. Huấn luyện

### 11.1 Một epoch

In [ ]:
def run_epoch(loader, train, net=None):
    """Trả về (loss trung bình theo MẪU, MAE theo MẪU cho [steer, longitudinal], MAE phụ).

    Bản cũ cộng dồn theo BATCH rồi chia số batch. Với val_loader không drop_last, batch cuối
    nhỏ hơn nhưng vẫn được tính trọng số ngang các batch đầy -> MAE lệch nhẹ và không so
    sánh được với baseline vốn tính trên toàn bộ mẫu. Ở đây cộng theo mẫu.

    v9: kéo thêm `aux` ra khỏi loader (v7 vứt nó đi bằng `_aux`) và cho model dự đoán nó.
    MAE phụ được trả về để §12 vẽ được — nếu đường này KHÔNG giảm thì nhánh CNN vẫn chưa học
    được gì về vị trí ngang, và mọi con số steer đẹp đẽ đều đáng ngờ.
    """
    net = model if net is None else net
    net.train(train)
    tot_loss, sum_abs, sum_aux_abs, n = 0.0, np.zeros(2), np.zeros(len(AUX_TARGET_COLS)), 0
    use_aux = AUX_LOSS_WEIGHT > 0
    with torch.enable_grad() if train else torch.no_grad():
        for masks, scalars, aux, targets in loader:
            masks = masks.to(DEVICE, non_blocking=True)
            scalars = scalars.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            aux_t = aux[:, AUX_TARGET_IDX].to(DEVICE, non_blocking=True)
            bs = targets.size(0)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                if use_aux:
                    preds, preds_aux = net(masks, scalars, return_aux=True)
                    loss = control_loss(preds, targets, preds_aux, aux_t)
                else:
                    preds, preds_aux = net(masks, scalars), None
                    loss = control_loss(preds, targets)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
                scheduler.step()            # lịch theo STEP, xem §10
                if ema is not None:
                    ema.update(net)

            tot_loss += loss.item() * bs
            sum_abs += (preds - targets).abs().sum(dim=0).detach().float().cpu().numpy()
            if preds_aux is not None:
                sum_aux_abs += (preds_aux - aux_t).abs().sum(dim=0).detach().float().cpu().numpy()
            n += bs
    return tot_loss / n, sum_abs / n, sum_aux_abs / n

### 11.2 Hàm lưu checkpoint

In [ ]:
# `val_aux_mae` là đường quan trọng nhất để theo dõi trong v9: nó đo trực tiếp xem nhánh CNN
# có đang học vị trí ngang không. Nếu nó PHẲNG trong khi `val_steer_mae` vẫn giảm thì model
# đang giảm loss bằng đường khác, không phải bằng cách nhìn ảnh.
HKEYS = ("train_loss", "val_loss", "val_steer_mae", "val_long_mae", "val_score",
         "val_aux_mae", "weights_src", "epoch_time", "lr")
history = {k: [] for k in HKEYS}
best_score, best_src, epochs_no_improve = float("inf"), "raw", 0
training_start = time.time()


def save_checkpoint(epoch, net, src, steer_mae, long_mae, score):
    torch.save({
        "epoch": epoch, "model_state_dict": net.state_dict(), "weights_source": src,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_score": score, "val_steer_mae": steer_mae, "val_long_mae": long_mae,
        # --- Hợp đồng nhãn + tiền xử lý + đặc trưng, để DRL không phải chép tay ---------
        "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
        "seg_label_lut": SEG_LABEL_LUT.tolist(),
        "road_id": ROAD_ID, "roadline_id": ROADLINE_ID, "sky_id": SKY_ID,
        "image_height": IMAGE_HEIGHT, "image_width": IMAGE_WIDTH,
        "seg_native_height": SEG_NATIVE_HEIGHT, "seg_native_width": SEG_NATIVE_WIDTH,
        "thin_cover_thresh": THIN_COVER_THRESH,
        # `pool_grid` PHẢI có mặt: nó quyết định shape của `cnn_fc.0.weight`, và
        # drl_training/policy/backbone.py đọc nó để tự kiểm tra trước khi warm-start.
        "pool_grid": list(POOL_GRID),
        "aux_target_cols": AUX_TARGET_COLS, "aux_loss_weight": float(AUX_LOSS_WEIGHT),
        "aux_target_scale": list(AUX_TARGET_SCALE),
        "normalize_action_loss": bool(NORMALIZE_ACTION_LOSS),
        "action_std": [float(ACTION_STD[0]), float(ACTION_STD[1])],
        # Danh sách cột đã CỐ Ý loại khỏi observation — để phía DRL đối chiếu được và không
        # ai vô tình thêm lại chúng vào `policy/observation.py`.
        "leaky_cols_excluded": LEAKY_COLS,
        "scalar_feature_dim": SCALAR_FEATURE_DIM,
        "scalar_feature_order": SCALAR_FEATURE_ORDER,
        "continuous_cols": CONTINUOUS_COLS, "raw_action_cols": RAW_ACTION_COLS,
        "use_prev_actions": bool(USE_PREV_ACTIONS),
        "norm_stats": norm_stats, "traffic_light_vocab": TRAFFIC_LIGHT_VOCAB,
        "collect_fps": COLLECT_FPS, "control_dt": CONTROL_DT,
        "train_towns": TRAIN_TOWNS, "val_towns": VAL_TOWNS,
        # --- Baseline đi kèm để không bao giờ phải nhớ lại "ngưỡng là bao nhiêu" -------
        "copycat_steer_mae": COPYCAT_STEER_MAE, "copycat_long_mae": COPYCAT_LONG_MAE,
        "const_steer_mae": CONST_STEER_MAE, "const_long_mae": CONST_LONG_MAE,
        # --- Siêu tham số của lần chạy, để đối chiếu với il_results.csv ----------------
        "run_name": RUN_NAME, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "huber_beta": HUBER_BETA, "sampler_mode": SAMPLER_MODE,
        "sampler_power": SAMPLER_POWER, "sampler_weight_cap": SAMPLER_WEIGHT_CAP,
        "drop_stationary_runs": bool(DROP_STATIONARY_RUNS),
        "seg_checkpoint": os.path.basename(SEG_CHECKPOINT_PATH),
        "trained_on_predicted_seg": bool(USE_PREDICTED_SEGMENTATION),
    }, STEER_CHECKPOINT_PATH)

### 11.3 Vòng lặp

In [ ]:
pbar = tqdm(range(EPOCHS), desc="Training IL")
for epoch in pbar:
    t0 = time.time()
    lr_now = optimizer.param_groups[0]["lr"]
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_mae, val_aux_mae = run_epoch(val_loader, False)
    if epoch == 0:
        print(f"  [hợp đồng] scalar_feature_order = {SCALAR_FEATURE_ORDER}")

    # Ứng viên: trọng số raw và trọng số EMA. Đánh giá EMA là MỘT lượt val đầy đủ nữa, chỉ
    # đáng làm sau khi trung bình trượt đã rời xa khởi tạo (EMA_EVAL_START).
    cands = [("raw", val_mae, val_loss)]
    if ema is not None and epoch >= EMA_EVAL_START:
        ema_loss, ema_mae, _ = run_epoch(val_loader, False, net=ema.module)
        cands.append(("ema", ema_mae, ema_loss))
    src, mae, vloss = min(cands, key=lambda c: val_score(c[1][0], c[1][1]))
    score = val_score(mae[0], mae[1])

    for k, v in zip(HKEYS, (train_loss, vloss, mae[0], mae[1], score,
                            float(np.mean(val_aux_mae)), src, time.time() - t0, lr_now)):
        history[k].append(v)
    pbar.set_postfix(train=f"{train_loss:.4f}", steer=f"{mae[0]:.4f}",
                     long=f"{mae[1]:.4f}", aux=f"{np.mean(val_aux_mae):.4f}", src=src)

    if score < best_score - 1e-6:
        best_score, best_src, epochs_no_improve = score, src, 0
        save_checkpoint(epoch, model if src == "raw" else ema.module,
                        src, float(mae[0]), float(mae[1]), float(score))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dừng sớm ở epoch {epoch+1}")
            break

_ck = torch.load(STEER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print(f"\nHoàn tất sau {(time.time()-training_start)/60:.1f} phút "
      f"({np.mean(history['epoch_time']):.1f}s/epoch) -> {STEER_CHECKPOINT_PATH}")
print(f"Best: epoch {_ck['epoch']+1} ({_ck['weights_source']}) "
      f"| steer MAE {_ck['val_steer_mae']:.4f} (baseline chép {COPYCAT_STEER_MAE:.4f}) "
      f"| long MAE {_ck['val_long_mae']:.4f} (baseline chép {COPYCAT_LONG_MAE:.4f})")
print(f"EMA thắng ở {sum(s == 'ema' for s in history['weights_src'])}/"
      f"{len(history['weights_src'])} epoch")
del _ck

> Checkpoint lưu kèm `norm_stats`, `scalar_feature_order`, `traffic_light_vocab` và
> `seg_label_lut` — lúc inference/demo phải nạp lại đúng các giá trị này, không tính lại từ
> dữ liệu mới. `drl_training/policy/observation.py` đọc thẳng các trường này để dựng scalar
> vector đúng thứ tự khi warm-start actor PPO.

## 12. Biểu đồ huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss theo epoch"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=.3)

# Hai đường ngang là thứ duy nhất cần nhìn: đường xanh phải chui xuống DƯỚI đường đỏ.
axes[1].plot(history["val_steer_mae"], label="Steer MAE", color="tab:blue")
axes[1].axhline(COPYCAT_STEER_MAE, color="r", ls=":", lw=1.4, label="chép prev (steer)")
axes[1].axhline(CONST_STEER_MAE, color="gray", ls="--", lw=1.0, label="hằng số (steer)")
axes[1].set_yscale("log")      # baseline hằng số cách baseline chép cả một bậc độ lớn
axes[1].set_title("Steer MAE vs baseline (Val, log)"); axes[1].set_xlabel("epoch")
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3, which="both")

axes[2].plot(history["val_long_mae"], label="Longitudinal MAE", color="tab:orange")
axes[2].axhline(COPYCAT_LONG_MAE, color="r", ls=":", lw=1.4, label="chép prev (long)")
axes[2].axhline(CONST_LONG_MAE, color="gray", ls="--", lw=1.0, label="hằng số (long)")
axes[2].set_yscale("log")
axes[2].set_title("Longitudinal MAE vs baseline (Val, log)"); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3, which="both")

plt.tight_layout();
plt.savefig('bieu_do_train.png', dpi=1500, bbox_inches='tight')
plt.show()

# Đường aux tách riêng: nó không cùng đơn vị với MAE hành động nên vẽ chung sẽ vô nghĩa.
# Đây là đường phải nhìn TRƯỚC khi tin vào val_steer_mae — xem chú thích ở §11.2.
if AUX_LOSS_WEIGHT > 0 and any(history["val_aux_mae"]):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(history["val_aux_mae"], color="tab:green")
    ax.set_title("MAE mục tiêu phụ (nhánh CNN suy ra vị trí ngang)")
    ax.set_xlabel("epoch"); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig('bieu_do_aux.png', dpi=300, bbox_inches='tight')
    plt.show()
    _first, _last = history["val_aux_mae"][0], history["val_aux_mae"][-1]
    print(f"aux MAE: {_first:.4f} -> {_last:.4f} "
          f"({100*(1-_last/max(_first,1e-9)):+.0f}%)")
    if _last > 0.8 * _first:
        print("[!] aux MAE gần như không giảm — nhánh CNN VẪN chưa học được vị trí ngang.")
        print("    Tăng AUX_LOSS_WEIGHT, hoặc kiểm tra lane_offset_m trong CSV có đúng không.")

# Epoch nào EMA thắng trọng số raw — nếu gần như mọi epoch thì nhiễu step-to-step còn lớn.
_ema_ep = [i + 1 for i, s in enumerate(history["weights_src"]) if s == "ema"]
print(f"Thời gian trung bình/epoch: {np.mean(history['epoch_time']):.1f}s "
      f"| LR cuối: {history['lr'][-1]:.2e}")
print(f"EMA được chọn ở epoch: {_ema_ep if _ema_ep else 'không epoch nào'}")

## 13. Đánh giá trên tập Validation

### 13.1 Chạy dự đoán

In [ ]:
ckpt = torch.load(STEER_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
eval_model = SteeringNet(num_scalar_features=ckpt["scalar_feature_dim"]).to(DEVICE)
eval_model.load_state_dict(ckpt["model_state_dict"]); eval_model.eval()
print(f"Đánh giá checkpoint epoch {ckpt['epoch']+1} ({ckpt['weights_source']}) "
      f"| run='{ckpt['run_name']}'")

all_preds, all_targets, all_aux = [], [], []
with torch.no_grad():
    for masks, scalars, aux, targets in tqdm(val_loader, desc="eval val"):
        preds = eval_model(masks.to(DEVICE), scalars.to(DEVICE))
        all_preds.append(preds.float().cpu().numpy())
        all_targets.append(targets.numpy()); all_aux.append(aux.numpy())
all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
all_aux     = np.concatenate(all_aux)      # cột: lane_offset_m, heading_error_rad, is_junction
errors = all_preds - all_targets
# val_loader không shuffle và không drop_last -> thứ tự trùng val_df.
all_tl   = val_df["traffic_light_state"].to_numpy()[:len(all_preds)]
prev_st  = val_df["previous_steer"].to_numpy()[:len(all_preds)]
mae_final = np.abs(errors).mean(axis=0)

### 13.2 Bảng ba baseline

In [ ]:
# =========================================================================================
# 1. BẢNG BA CỘT — con số chính của báo cáo
# =========================================================================================
print("\n" + "=" * 74)
print(f"{'':<14}{'model':>10}{'chép prev':>12}{'hằng số':>10}{'vs chép':>10}{'vs hằng số':>12}")
for i, (lb, cc, cst) in enumerate((("Steer", COPYCAT_STEER_MAE, CONST_STEER_MAE),
                                   ("Longitudinal", COPYCAT_LONG_MAE, CONST_LONG_MAE))):
    g1 = 100 * (1 - mae_final[i] / cc) if cc > 0 else float("nan")
    g2 = 100 * (1 - mae_final[i] / cst) if cst > 0 else float("nan")
    print(f"{lb:<14}{mae_final[i]:>10.4f}{cc:>12.4f}{cst:>10.4f}"
          f"{g1:>+9.1f}%{g2:>+11.1f}%   " + ("ĐẠT" if mae_final[i] < cc else "KHÔNG ĐẠT"))
print("=" * 74)
print("'chép prev' là ngưỡng QUYẾT ĐỊNH. 'hằng số' chỉ cho thấy độ khó cơ bản của bài toán.")

### 13.3 Model thắng/thua ở đâu

In [ ]:
# =========================================================================================
# 2. MODEL THẮNG/THUA Ở ĐÂU — chia theo mức thay đổi thật của vô-lăng
# =========================================================================================
# Sai số của baseline "chép" trên MỖI mẫu chính xác bằng |steer - previous_steer|. Chia
# theo đại lượng đó cho thấy điều mà MAE trung bình che mất: baseline gần như hoàn hảo khi
# xe đi thẳng, và sụp đổ khi vô-lăng thật sự đổi — đúng lúc bộ điều khiển cần đúng.
_d = np.abs(all_targets[:, 0] - prev_st)
cmp_tab = pd.DataFrame({
    "bin": pd.cut(_d, [-1e-9, 0.005, 0.02, 0.05, np.inf],
                  labels=["<0.005", "0.005-0.02", "0.02-0.05", ">0.05"]),
    "model": np.abs(errors[:, 0]),
    "chép prev": _d,
}).groupby("bin", observed=False).agg(
    n=("model", "size"), model=("model", "mean"), copy_=("chép prev", "mean"))
cmp_tab["model thắng"] = np.where(cmp_tab["model"] < cmp_tab["copy_"], "có", "không")
print("\nMAE steer theo |steer - previous_steer| (sai số của baseline chính là cột này):")
print(cmp_tab.rename(columns={"copy_": "chép prev"}).round(4).to_string())

### 13.4 Lỗi nguy hiểm

In [ ]:
# =========================================================================================
# 3. LỖI NGUY HIỂM — thứ MAE trung bình không nhìn thấy
# =========================================================================================
# Một frame "phanh gấp mà model đạp ga" có |err| ~ 1.5 nhưng nếu chỉ chiếm 1% mẫu thì nó
# chỉ đóng góp 0.015 vào MAE — chìm nghỉm. Trong vòng kín thì đó là một va chạm.
gt_l, pr_l = all_targets[:, 1], all_preds[:, 1]
brake_as_throttle = int(((gt_l < -0.1) & (pr_l > 0.1)).sum())
throttle_as_brake = int(((gt_l > 0.1) & (pr_l < -0.1)).sum())
sign_flip_steer = int(((np.abs(all_targets[:, 0]) > 0.05) &
                       (np.sign(all_preds[:, 0]) != np.sign(all_targets[:, 0]))).sum())
N = len(all_preds)
print("\nLỗi nguy hiểm (không hiện ra trong MAE):")
print(f"  GT phanh  -> model ĐẠP GA : {brake_as_throttle:>5} / {N}  "
      f"({100*brake_as_throttle/N:.2f}%)   <-- nguy hiểm nhất")
print(f"  GT ga     -> model PHANH  : {throttle_as_brake:>5} / {N}  "
      f"({100*throttle_as_brake/N:.2f}%)")
print(f"  steer NGƯỢC dấu (|gt|>.05): {sign_flip_steer:>5} / {N}  "
      f"({100*sign_flip_steer/N:.2f}%)")
print(f"  steer p99 = {np.percentile(np.abs(errors[:,0]), 99):.4f} "
      f"| max = {np.abs(errors[:,0]).max():.4f}")
print(f"  long  p99 = {np.percentile(np.abs(errors[:,1]), 99):.4f} "
      f"| max = {np.abs(errors[:,1]).max():.4f}")

### 13.5 Biểu đồ & bảng chẩn đoán

In [ ]:
# =========================================================================================
# 4. Biểu đồ
# =========================================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, lb in enumerate(["Steer", "Longitudinal"]):
    axes[i].scatter(all_targets[:, i], all_preds[:, i], alpha=0.25, s=8)
    lims = [min(all_targets[:, i].min(), all_preds[:, i].min()),
            max(all_targets[:, i].max(), all_preds[:, i].max())]
    axes[i].plot(lims, lims, "r--", lw=1)
    if i == 1:      # tô hai góc phần tư "đổi dấu" — vùng gây va chạm
        axes[i].axhspan(0.1, lims[1], xmin=0, xmax=0.45, color="red", alpha=0.06)
    axes[i].set_xlabel("Thực tế"); axes[i].set_ylabel("Dự đoán"); axes[i].set_title(lb)
plt.tight_layout()
plt.savefig('bieu_do_val.png', dpi=1500, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, lb in enumerate(["Steer", "Longitudinal"]):
    axes[i].hist(errors[:, i], bins=60)
    axes[i].set_yscale("log")     # đuôi hiếm mới là phần đáng lo, thang tuyến tính giấu nó
    axes[i].set_title(f"Phân bố sai số {lb} (log)")
plt.tight_layout()
plt.savefig('bieu_do_val1.png', dpi=1500, bbox_inches='tight')
plt.show()

mae_by_tl = pd.DataFrame({"tl": all_tl, "steer_ae": np.abs(errors[:, 0]),
                          "long_ae": np.abs(errors[:, 1])})
mae_by_tl = mae_by_tl.groupby("tl")[["steer_ae", "long_ae"]].mean().reindex(TRAFFIC_LIGHT_VOCAB)
print("\nMAE theo trạng thái đèn:")
print(mae_by_tl.round(4).to_string())
if mae_by_tl["steer_ae"].isna().any():
    print("[!] Trạng thái đèn có NaN = val không có mẫu nào. Không kết luận gì về nó trong "
          "báo cáo.")

# aux KHÔNG vào model, chỉ dùng ở đây. Kỳ vọng: steer_ae tăng khi |lane_offset_m| lớn — đó
# là vùng phải lái mạnh để hồi phục, và là vùng cần thêm dữ liệu recovery/DAgger.
_off = np.abs(all_aux[:, 0])
mae_by_offset = pd.DataFrame({
    "bin": pd.cut(_off, bins=[-1e-9, 0.15, 0.4, 0.8, np.inf],
                  labels=["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"]),
    "steer_ae": np.abs(errors[:, 0]),
}).groupby("bin", observed=False)["steer_ae"].agg(["mean", "count"])
print("\nMAE steer theo |lane_offset_m| (chỉ để chẩn đoán):")
print(mae_by_offset.round(4).to_string())
RECOVERY_MAE = float(np.abs(errors[_off > 0.15, 0]).mean()) if (_off > 0.15).any() else float("nan")
CENTER_MAE   = float(np.abs(errors[_off <= 0.15, 0]).mean())
print(f"MAE giữa làn (<0.15m) = {CENTER_MAE:.4f} | MAE lệch làn (>0.15m) = {RECOVERY_MAE:.4f}"
      f"  -> gấp {RECOVERY_MAE/max(CENTER_MAE,1e-9):.1f} lần")
print(f"Chỉ {100*float((_off > 0.15).mean()):.1f}% mẫu val nằm ngoài 0.15m — autopilot luôn "
      "chạy giữa làn,\nnên model chưa từng học cách quay về. Đây là chỗ cần dữ liệu recovery.")

### 13.6 Nhật ký thí nghiệm & kết luận

In [ ]:
# =========================================================================================
# 5. Ghi một dòng vào nhật ký thí nghiệm
# =========================================================================================
_row = {
    "run": RUN_NAME, "time": pd.Timestamp.now().strftime("%m-%d %H:%M"),
    "n_train": len(train_df), "n_val": len(val_df),
    "batch": BATCH_SIZE, "lr": round(LEARNING_RATE, 6), "huber_beta": HUBER_BETA,
    "sampler": SAMPLER_MODE, "sampler_power": SAMPLER_POWER,
    "prev_actions": USE_PREV_ACTIONS,
    "drop_stationary": DROP_STATIONARY_RUNS, "pred_seg": USE_PREDICTED_SEGMENTATION,
    "best_epoch": int(ckpt["epoch"]) + 1, "src": ckpt["weights_source"],
    "steer_mae": round(float(mae_final[0]), 5),
    "long_mae": round(float(mae_final[1]), 5),
    "copy_steer": round(COPYCAT_STEER_MAE, 5), "copy_long": round(COPYCAT_LONG_MAE, 5),
    "const_steer": round(CONST_STEER_MAE, 5), "const_long": round(CONST_LONG_MAE, 5),
    "beats_copy": bool(mae_final[0] < COPYCAT_STEER_MAE),
    "brake_as_throttle": brake_as_throttle,
    "recovery_mae": round(RECOVERY_MAE, 5), "center_mae": round(CENTER_MAE, 5),
}
_log = pd.DataFrame([_row])
if os.path.exists(RESULTS_LOG):
    _prev = pd.read_csv(RESULTS_LOG)
    if (_prev["run"] == RUN_NAME).any():
        print(f"[!] RUN_NAME '{RUN_NAME}' đã có trong nhật ký. Đổi tên mỗi lần chạy, kẻo")
        print("    bảng so sánh trong báo cáo có hai dòng trùng bị đọc thành hai thí nghiệm.")
    _log = pd.concat([_prev, _log], ignore_index=True)
_log.to_csv(RESULTS_LOG, index=False)
print(f"\nĐã ghi kết quả vào {RESULTS_LOG} ({len(_log)} lần chạy):")
print(_log[["run", "steer_mae", "copy_steer", "beats_copy", "long_mae",
            "brake_as_throttle"]].to_string(index=False))

# =========================================================================================
# 6. Kết luận
# =========================================================================================
print("\n" + "=" * 74)
if mae_final[0] < COPYCAT_STEER_MAE and mae_final[1] < COPYCAT_LONG_MAE:
    print(f"ĐẠT — model IL sẵn sàng warm-start cho DRL: {STEER_CHECKPOINT_PATH}")
    print(f"  Nhớ: DRL phải dựng scalar vector đúng thứ tự {SCALAR_FEATURE_ORDER}")
    print(f"  và đặt fixed_delta_seconds = {CONTROL_DT} ({COLLECT_FPS:.0f} FPS).")
else:
    print("[!] CHƯA đạt. Gợi ý theo ĐÚNG đầu ra nào đang hỏng:")
    if mae_final[0] >= COPYCAT_STEER_MAE:
        print("  STEER thua baseline:")
        print("    1. SAMPLER_MODE = 'offset'   — 13.8% mẫu lệch làn gây phần lớn sai số")
        print("    2. USE_PREV_ACTIONS = False  — buộc model đọc ảnh thay vì phát lại input")
    if mae_final[1] >= COPYCAT_LONG_MAE:
        print("  LONGITUDINAL thua baseline:")
        print("    1. DROP_STATIONARY_RUNS = False — cờ này bỏ luôn pha phanh khỏi train")
        print("    2. USE_PREV_ACTIONS = True      — mask segmentation KHÔNG chứa đèn tín")
        print("       hiệu hay xe khác (chỉ 4 lớp đường/vạch/vỉa hè), nên ga-phanh gần như")
        print("       không suy ra được từ ảnh. previous_longitudinal là tín hiệu chính.")
        print("    3. Kiểm dòng 'Lệch phân phối train/val' ở §6 trước khi đổi gì khác.")
    print("  Đổi RUN_NAME mỗi lần; il_results.csv gom thành bảng so sánh cho báo cáo.")
print("=" * 74)

## 14. Kiểm tra sẵn sàng warm-start cho DRL

In [ ]:
# =========================================================================================
# §14. KIỂM TRA SẴN SÀNG WARM-START — chạy trước khi mang checkpoint sang DRL
# =========================================================================================
# Mọi thứ dưới đây là lỗi IM LẶNG nếu sai: shape vẫn khớp, không exception nào, chỉ có
# policy hành xử vô nghĩa sau vài nghìn step và không ai biết vì sao.
ck = torch.load(STEER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
ready, blockers, warnings_ = True, [], []

print("HỢP ĐỒNG DRL PHẢI ĐỌC TỪ CHECKPOINT (đừng chép tay):")
print(f"  scalar_feature_order = {ck['scalar_feature_order']}")
print(f"  scalar_feature_dim   = {ck['scalar_feature_dim']}")
print(f"  traffic_light_vocab  = {ck['traffic_light_vocab']}")
print(f"  norm_stats           = "
      + ", ".join(f"{k}({v[0]:.3f},{v[1]:.3f})" for k, v in ck["norm_stats"].items()))
print(f"  control_dt           = {ck['control_dt']}  -> fixed_delta_seconds của CARLA")
print(f"  pool_grid            = {ck['pool_grid']}  -> POOL_GRID của policy/backbone.py")
print(f"  cột đã loại (rò rỉ)  = {ck['leaky_cols_excluded']}")
print(f"  class_names          = {ck['class_names']} @ "
      f"{ck['image_height']}x{ck['image_width']} (gốc {ck['seg_native_height']}x"
      f"{ck['seg_native_width']}, thin_cover={ck['thin_cover_thresh']})")

# 1. Ngưỡng quyết định.
if not (ck["val_steer_mae"] < ck["copycat_steer_mae"]):
    ready = False
    blockers.append(
        f"steer MAE {ck['val_steer_mae']:.5f} KHÔNG thấp hơn baseline chép "
        f"{ck['copycat_steer_mae']:.5f}. Nạp checkpoint này vào actor còn TỆ HƠN khởi tạo "
        f"ngẫu nhiên: model đã học ánh xạ 'output ~ previous_steer', một điểm hút mà PPO/SAC "
        f"phải phá bỏ trước khi học được gì.")

# 1b. Khớp kiến trúc với DRL. Đây là lỗi đã CHẶN checkpoint v8: pool (1,1) cho cnn_fc 64
#     chiều, trong khi policy/backbone.py đã chuyển sang POOL_GRID (4,6) = 1536 chiều.
_cnn_in = ck["model_state_dict"]["cnn_fc.0.weight"].shape[1]
_expect = 64 * ck["pool_grid"][0] * ck["pool_grid"][1]
if _cnn_in != _expect:
    ready = False
    blockers.append(f"cnn_fc nhận {_cnn_in} chiều nhưng pool_grid {ck['pool_grid']} đòi "
                    f"{_expect}. Checkpoint hỏng.")
print(f"\n  cnn_fc.0.weight = {_cnn_in} chiều vào — drl_training/policy/backbone.py PHẢI "
      f"có POOL_GRID = {tuple(ck['pool_grid'])}")

# 1c. Rò rỉ quan sát: cột nào trong LEAKY_COLS lọt lại vào observation?
_leaked = [c for c in ck["leaky_cols_excluded"] if c in ck["scalar_feature_order"]]
if _leaked:
    ready = False
    blockers.append(f"{_leaked} vẫn nằm trong observation. Đây là các đại lượng đo CÙNG bước "
                    f"thời gian với hành động — model sẽ đọc thẳng đáp án thay vì nhìn ảnh, "
                    f"và trong vòng kín chúng tạo vòng lặp tự duy trì (xem §2.3).")

# 1d. Nhánh CNN có thật sự học vị trí ngang không? aux MAE phẳng = chưa học.
if AUX_LOSS_WEIGHT > 0 and len(history["val_aux_mae"]) > 1:
    _a0, _a1 = history["val_aux_mae"][0], history["val_aux_mae"][-1]
    if _a1 > 0.8 * _a0:
        warnings_.append(f"aux MAE chỉ đi từ {_a0:.4f} xuống {_a1:.4f} — nhánh CNN gần như "
                         f"chưa mã hoá được vị trí ngang. Warm-start vẫn dùng được nhưng "
                         f"phần 'nhìn ảnh' của nó yếu; cân nhắc tăng AUX_LOSS_WEIGHT.")

# 2. Ô one-hot chết: chiều nào của scalar_mlp gần như không được huấn luyện?
_w = ck["model_state_dict"]["scalar_mlp.0.weight"].abs().mean(dim=0)
for _i, _name in enumerate(ck["scalar_feature_order"]):
    if _w[_i] < 0.02 * _w.mean():
        warnings_.append(f"'{_name}' có trọng số vào scalar_mlp gần như bằng 0 — nhiều khả "
                         f"năng đặc trưng này không bao giờ bật trong dữ liệu train.")

# 3. Vùng lệch làn — nơi vòng kín thật sự sống hay chết.
if not np.isnan(RECOVERY_MAE) and RECOVERY_MAE > 3 * CENTER_MAE:
    warnings_.append(
        f"MAE lệch làn ({RECOVERY_MAE:.4f}) gấp {RECOVERY_MAE/CENTER_MAE:.1f} lần MAE giữa "
        f"làn ({CENTER_MAE:.4f}). Policy sẽ ổn khi đi đúng và lạc lối ngay khi trôi -> sai "
        f"số tự khuếch đại. Ưu tiên SAMPLER_MODE='offset' hoặc thu thêm pha recovery.")

# 4. Lệch phân phối seg giữa lúc train IL và lúc chạy DRL.
#    KHÔNG có lệch trong pipeline hiện tại: cả collector (carla_collector/sensors.py) lẫn
#    env DRL (drl_training/envs/carla_lane_keep_env.py -> "sensor.camera.semantic_segmentation")
#    lẫn dashboard đều đọc camera segmentation GROUND-TRUTH của CARLA. Model segmentation
#    KHÔNG chạy ở bất kỳ đâu trong đường này — nó chỉ được dùng nếu bật
#    USE_PREDICTED_SEGMENTATION ở §2, tức cố tình train IL trên mask dự đoán.
if ck["trained_on_predicted_seg"]:
    warnings_.append("IL học trên mask DỰ ĐOÁN nhưng env DRL đọc camera segmentation "
                     "GROUND-TRUTH của CARLA -> có lệch phân phối theo chiều ngược lại. "
                     "Hoặc đặt USE_PREDICTED_SEGMENTATION = False, hoặc cho env dùng cùng "
                     "model seg.")
else:
    print("\n  [i] IL train trên mask GROUND-TRUTH, và env DRL cũng đọc camera segmentation")
    print("      ground-truth của CARLA -> KHÔNG có lệch phân phối. Model segmentation không")
    print("      tham gia đường chạy này (xem drl_training/demo_il.py).")

# 5. Lỗi đổi dấu.
if brake_as_throttle > 0.005 * len(all_preds):
    warnings_.append(f"{brake_as_throttle} frame 'GT phanh -> model đạp ga' "
                     f"({100*brake_as_throttle/len(all_preds):.2f}%). MAE không thấy nhưng "
                     f"trong vòng kín đây là va chạm.")

print("\n" + "=" * 74)
for b in blockers:
    print("CHẶN   : " + b.replace("\n", "\n         "))
for w in warnings_:
    print("LƯU Ý  : " + w.replace("\n", "\n         "))
if ready:
    print("\nSẴN SÀNG warm-start. Ba thứ phải làm đúng bên DRL:")
    print("  1. fixed_delta_seconds = {:.2f}  (previous_* nghĩa là 'lệnh của {:.0f}ms trước')"
          .format(ck["control_dt"], 1000 * ck["control_dt"]))
    print("  2. lane_offset_m / heading_error_rad / is_junction CHỈ dùng cho REWARD, tuyệt")
    print("     đối không đưa vào observation — notebook cố tình tách chúng ra tên 'aux'.")
    print("  3. Critic khởi tạo ngẫu nhiên sẽ sinh advantage nhiễu và XOÁ trọng số IL trong")
    print("     vài trăm update đầu. Bắt buộc: warmup critic (đóng băng actor), actor LR")
    print("     1e-5..3e-5 lúc đầu, và log_std khởi tạo nhỏ (~-1.0).")
else:
    print("\nCHƯA sẵn sàng. Thử theo thứ tự, mỗi lần ĐỔI MỘT thứ và đổi RUN_NAME:")
    print("  1. AUX_LOSS_WEIGHT 0.5 -> 1.0   — ép nhánh CNN mã hoá vị trí ngang mạnh hơn")
    print("  2. SAMPLER_MODE = 'offset'      — nhắm thẳng vào vùng lệch làn, không cần thu lại")
    print("  3. DROP_STATIONARY_RUNS         — bật/tắt và xem §6.2: tỉ lệ phanh train phải")
    print("     bám sát val, lệch nhiều là hỏng longitudinal theo một trong hai hướng.")
    print("  LƯU Ý: với USE_PREV_ACTIONS = False, MAE model TỆ HƠN baseline 'chép' là chuyện")
    print("  bình thường — baseline đó được phép nhìn previous_steer còn model thì không.")
    print("  Thước đo thật là §14 + demo vòng kín (drl_training/demo_il.py), không phải MAE.")
print("=" * 74)
del ck

In [ ]:
# §15. XUẤT GÓI BÀN GIAO CHO DRL
# =========================================================================================
# Máy DRL chạy CARLA 0.9.10 -> Python 3.7 -> PyTorch <= 1.13, trong khi notebook này chạy
# torch 2.x. File .pth pickle bằng torch 2.x KHÔNG chắc nạp được ngược. Xuất sang npz + json
# là định dạng trung tính, miễn nhiễm với chênh lệch phiên bản.
import json, shutil, zipfile
 
BUNDLE_DIR = "/kaggle/working/drl_bundle"
os.makedirs(BUNDLE_DIR, exist_ok=True)
 
_ck = torch.load(STEER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
 
# --- 1. Trọng số, dạng thuần numpy -------------------------------------------------------
np.savez(os.path.join(BUNDLE_DIR, "il_weights.npz"),
         **{k: v.detach().cpu().numpy() for k, v in _ck["model_state_dict"].items()})
 
# --- 2. Hợp đồng ------------------------------------------------------------------------
# Ba nhóm: (a) cách dựng observation, (b) cách diễn giải action, (c) cách dựng lại mạng.
# Thiếu bất kỳ nhóm nào là policy chạy sai mà KHÔNG có exception nào báo.
contract = {
    # (a) observation
    "scalar_feature_order": _ck["scalar_feature_order"],
    "scalar_feature_dim":   _ck["scalar_feature_dim"],
    "continuous_cols":      _ck["continuous_cols"],
    "raw_action_cols":      _ck["raw_action_cols"],
    "use_prev_actions":     _ck["use_prev_actions"],
    "norm_stats":           {k: list(v) for k, v in _ck["norm_stats"].items()},
    "traffic_light_vocab":  _ck["traffic_light_vocab"],
    # green đã bị gộp -> DRL phải map lại, nếu không one-hot toàn 0 ở mọi đèn xanh.
    "traffic_light_alias":  {"green": "unknown"} if MERGE_GREEN_INTO_UNKNOWN else {},
 
    # (b) segmentation: DRL dùng semantic camera CỦA CARLA, không cần model seg.
    #     Chỉ cần đúng LUT + đúng cách hạ mẫu là mask khớp phân phối lúc train IL.
    "seg_label_lut":     _ck["seg_label_lut"],
    "class_names":       _ck["class_names"],
    "num_classes":       _ck["num_classes"],
    "seg_native_height": _ck["seg_native_height"],
    "seg_native_width":  _ck["seg_native_width"],
    "image_height":      _ck["image_height"],
    "image_width":       _ck["image_width"],
    "thin_cover_thresh": _ck["thin_cover_thresh"],
    "thin_class_ids":    [_ck["roadline_id"]],
 
    # (c) điều khiển. `longitudinal` là MỘT số trong [-1,1]: dương = ga, âm = phanh.
    #     Không ghi lại quy ước này thì phía DRL rất dễ áp nhầm thành throttle-only.
    "control_dt":            _ck["control_dt"],
    "collect_fps":           _ck["collect_fps"],
    "action_space":          ["steer", "longitudinal"],
    "action_range":          [-1.0, 1.0],
    "longitudinal_positive": "throttle",
    "longitudinal_negative": "brake",
    "carla_version":         "0.9.10",
    # 0.9.10 chưa có physics substepping -> tick 0.2s bị mô phỏng thành MỘT bước vật lý.
    # Chạy sim ở 0.05s và lặp lại action 4 lần để giữ đúng control_dt.
    "sim_delta_seconds": 0.05,
    "action_repeat":     int(round(_ck["control_dt"] / 0.05)),
 
    # (d) kiến trúc — để dựng lại SteeringNet mà không phải chép tay
    "arch": {"name": "SteeringNet", "conv_channels": [24, 36, 48, 64, 64],
             "pool_grid": _ck["pool_grid"], "cnn_fc": 64, "scalar_mlp": [32, 32],
             "head": [64, 32, 2], "output_activation": "tanh",
             # aux_head CHỈ có ở IL. drl_training/policy/il_compat.py lọc theo tiền tố nên
             # nó bị bỏ qua lúc warm-start — ghi ra đây để không ai tưởng là thiếu sót.
             "aux_head": [32, len(_ck["aux_target_cols"])],
             "aux_target_cols": _ck["aux_target_cols"]},
    "leaky_cols_excluded": _ck["leaky_cols_excluded"],
 
    # (e) ngưỡng đã đạt, để phía DRL biết mình đang khởi tạo từ cái gì
    "val_steer_mae":     _ck["val_steer_mae"],
    "val_long_mae":      _ck["val_long_mae"],
    "copycat_steer_mae": _ck["copycat_steer_mae"],
    "copycat_long_mae":  _ck["copycat_long_mae"],
    "beats_copy_steer":  bool(_ck["val_steer_mae"] < _ck["copycat_steer_mae"]),
    "beats_copy_long":   bool(_ck["val_long_mae"] < _ck["copycat_long_mae"]),
    "trained_on_predicted_seg": _ck["trained_on_predicted_seg"],
    "run_name": _ck["run_name"], "epoch": int(_ck["epoch"]) + 1,
}
with open(os.path.join(BUNDLE_DIR, "il_contract.json"), "w", encoding="utf-8") as f:
    json.dump(contract, f, ensure_ascii=False, indent=2)
 
# --- 3. Kiểm tra vòng tròn: dựng lại từ npz và so khớp bit ------------------------------
_z = np.load(os.path.join(BUNDLE_DIR, "il_weights.npz"))
_probe = SteeringNet(num_scalar_features=contract["scalar_feature_dim"],
                    pool_grid=tuple(_ck["pool_grid"]),
                    num_aux=len(_ck["aux_target_cols"]))
_probe.load_state_dict({k: torch.from_numpy(_z[k]) for k in _z.files})
_probe.eval()
with torch.no_grad():
    _m = torch.zeros(2, IMAGE_HEIGHT, IMAGE_WIDTH, dtype=torch.long)
    _s = torch.zeros(2, contract["scalar_feature_dim"])
    _out = _probe(_m, _s)
np.save(os.path.join(BUNDLE_DIR, "smoke_test.npy"),
        np.concatenate([_m.numpy().reshape(2, -1)[:, :4], _s.numpy(), _out.numpy()], 1))
print(f"Kiểm tra nạp lại: output trên input rỗng = {_out[0].tolist()}")
print("  Bên DRL chạy lại đúng phép này; lệch quá 1e-5 nghĩa là dựng sai kiến trúc.")
 
for f in ["il_results.csv"]:
    if os.path.exists(f"/kaggle/working/{f}"):
        shutil.copy(f"/kaggle/working/{f}", BUNDLE_DIR)
 
with zipfile.ZipFile("/kaggle/working/drl_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(BUNDLE_DIR)):
        z.write(os.path.join(BUNDLE_DIR, f), f)
 
print(f"\nGói bàn giao -> /kaggle/working/drl_bundle.zip")
for f in sorted(os.listdir(BUNDLE_DIR)):
    print(f"  {f:24} {os.path.getsize(os.path.join(BUNDLE_DIR, f))/1024:8.1f} KB")
print(f"\nsim_delta_seconds={contract['sim_delta_seconds']} x "
      f"action_repeat={contract['action_repeat']} = control_dt {contract['control_dt']}")
del _ck, _z, _probe